# Занятие 5. SVM, снижение размерности и кластеризация

Занятие состоит из четырех частей.

**Метод опорных векторов** (support vector machine, SVM). Линейный
классификатор, который выбирает разделяющую прямую с наибольшей шириной
полосы между классами. Затем нелинейный вариант: ядра позволяют строить
криволинейные границы без явного построения новых признаков.

**Метод главных компонент** (principal component analysis, PCA). Переход
к направлениям наибольшей дисперсии данных. Метод выводится из задачи
максимизации дисперсии проекций и применяется к изображениям цифр
MNIST (Modified National Institute of Standards and Technology database)
размером 28 × 28 пикселей.

**Визуализация многомерных данных.** Методы t-SNE (t-distributed stochastic
neighbor embedding, стохастическое вложение соседей с распределением
Стьюдента) и UMAP (uniform manifold approximation and projection,
равномерная аппроксимация и проекция многообразия) строят двумерное
представление данных, в котором сохраняются ближайшие соседи.

**Кластеризация.** Разбиение объектов на группы без известных ответов:
метрики качества, метод k-средних (k-means), агломеративная кластеризация
(agglomerative clustering), DBSCAN
(density-based spatial clustering of applications with noise, плотностная
кластеризация с выделением шума), смесь гауссовых распределений
(Gaussian mixture model, GMM) и их
сравнение на одних наборах данных.

Весь код в ноутбуке написан, ячейки выполняют по порядку. При первом
запуске нужен интернет: набор MNIST (около 50 МБ) загружается с OpenML
и далее хранится в кэше.

## 0. Подготовка

Функция `show_table` раскрашивает числовые столбцы таблицы: лучшее значение
выделено самым ярким цветом. Для столбцов, где лучше меньшее значение
(время, индекс Дэвиса — Болдина), шкала обращена; для таблицы, где все
значения такого рода, задают `lower_is_better=True`. При `axis=None` у всей
таблицы одна шкала.

In [ ]:
import itertools
import time
import warnings
from math import comb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import umap
import umap.umap_ as umap_internal
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.patches import Circle, Ellipse, Rectangle
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import fetch_openml, load_wine, make_blobs, make_circles, make_moons
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.manifold import TSNE, trustworthiness
from sklearn.metrics import (
    adjusted_rand_score, calinski_harabasz_score, completeness_score, davies_bouldin_score,
    homogeneity_score, normalized_mutual_info_score, silhouette_score, v_measure_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVC, LinearSVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
CLUSTER_COLORS = np.array(["#0072CE", "#C98A3C", "#4DAF4A", "#984EA3", "#E41A1C",
                           "#FF7F00", "#A65628", "#F781BF", "#17BECF", "#6B7280"])
SEED = 42
LOWER_IS_BETTER = ("время", "секунд", "Дэвис")


def show_table(frame, digits=3, axis=0, lower_is_better=False):
    """Таблица с заливкой: лучшее значение ярче; для времени и индекса Дэвиса — Болдина лучше меньшее."""
    numeric = list(frame.select_dtypes("number").columns)
    lower = [c for c in numeric if lower_is_better or any(key in str(c) for key in LOWER_IS_BETTER)]
    higher = [c for c in numeric if c not in lower]
    styled = frame.style.format(precision=digits, subset=numeric)
    if higher:
        styled = styled.background_gradient(cmap="Greens", subset=higher, axis=axis)
    if lower:
        styled = styled.background_gradient(cmap="Greens_r", subset=lower, axis=axis)
    return styled

## 1. Метод опорных векторов (SVM)

### Отступ и функции потерь

Используются данные о винах из третьего занятия: сорт 1 против остальных,
два стандартизованных признака — флавоноиды и интенсивность цвета. Метки
классов кодируют как $y \in \{-1, +1\}$. Линейный классификатор вычисляет
$f(x) = \langle w, x \rangle + b$ и относит объект к классу $\mathrm{sign}\, f(x)$.

**Отступ** (margin) объекта:

$$M = y \cdot \big(\langle w, x \rangle + b\big).$$

При $M > 0$ объект классифицирован верно, при $M < 0$ — с ошибкой; модуль $M$
пропорционален расстоянию до границы. Функции потерь линейных
классификаторов записывают через отступ:

- доля ошибок (misclassification rate): $[M < 0]$;
- квадратичная функция потерь (squared loss) линейной регрессии на метках: $(1 - M)^2$;
- логистическая функция потерь (logistic loss): $\log_2(1 + e^{-M})$;
- кусочно-линейная функция потерь метода опорных векторов (hinge loss): $\max(0, 1 - M)$.

In [ ]:
margin_grid = np.linspace(-2.5, 3, 500)
loss_curves = {
    "доля ошибок, [M < 0]": ((margin_grid < 0).astype(float), GREY, 2),
    "квадратичная, (1 − M)²": ((1 - margin_grid) ** 2, BLACK, 1.5),
    "логистическая, log₂(1 + e⁻ᴹ)": (np.log2(1 + np.exp(-margin_grid)), BLUE, 2),
    "hinge loss, max(0, 1 − M)": (np.maximum(0, 1 - margin_grid), OCHRE, 3),
}
plt.figure(figsize=(8.5, 4.5))
for name, (values, color, width) in loss_curves.items():
    plt.plot(margin_grid, values, color=color, lw=width, label=name)
plt.axvspan(1, 3, color=OCHRE, alpha=0.08)
plt.text(1.1, 3.3, "M ≥ 1: hinge loss\nравна нулю", color=OCHRE, fontsize=10)
plt.axvline(0, color=GREY, lw=0.8)
plt.ylim(0, 4)
plt.xlabel("отступ M")
plt.ylabel("потери на объекте")
plt.legend(loc="upper right", fontsize=9)
plt.show()

Квадратичная функция потерь возрастает и при $M > 1$, то есть штрафует
объекты, классифицированные верно с большим запасом. Логистическая функция
потерь положительна при любом $M$: каждый объект вносит ненулевой вклад.
Hinge loss равна нулю при $M \geq 1$, поэтому объекты, лежащие дальше
единичного отступа, на решение не влияют. Это свойство приводит к понятию
опорных векторов.

In [ ]:
wine = load_wine(as_frame=True).frame
y_wine = (wine["target"] == 1).astype(int).to_numpy()
X_wine_raw = wine[["flavanoids", "color_intensity"]].to_numpy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_wine_raw, y_wine, test_size=0.3, random_state=SEED, stratify=y_wine
)
wine_scaler = StandardScaler().fit(X_train_raw)
X_train = wine_scaler.transform(X_train_raw)
X_test = wine_scaler.transform(X_test_raw)
print("обучение:", X_train.shape, " тестовый сет:", X_test.shape)

In [ ]:
def plot_points(ax, X, y, alpha=0.8):
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=26, color=GREY, alpha=alpha, label="другие сорта")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=26, color=BLUE, alpha=alpha, label="сорт 1")


def boundary_line(ax, X, w, b, color=BLACK, label=None, band=False, lw=2):
    xs = np.linspace(X[:, 0].min() - 0.3, X[:, 0].max() + 0.3, 100)
    ax.plot(xs, -(w[0] * xs + b) / w[1], color=color, lw=lw, label=label)
    if band:
        for shift in (-1, 1):
            ax.plot(xs, -(w[0] * xs + b - shift) / w[1], color=color, lw=1, ls="--")


def margins_of(X, y, w, b):
    margins = (2 * y - 1) * (X @ w + b)
    return margins

### Разделяющая полоса наибольшей ширины

Если классы разделимы, прямых без ошибок бесконечно много. Прямая, проходящая
вблизи объектов одного из классов, начинает ошибаться при малом сдвиге
данных. Поэтому выбирают прямую с наибольшим **зазором** (margin width) — расстоянием
до ближайшего объекта обучающей выборки. На рисунке две произвольные
прямые без ошибок и прямая метода опорных векторов на двух разделимых облаках.

In [ ]:
rng_svm = np.random.default_rng(7)
cloud_a = rng_svm.normal([0, 0], 0.7, (40, 2))
cloud_b = rng_svm.normal([3, 3], 0.7, (40, 2))
X_sep = np.vstack([cloud_a, cloud_b])
y_sep = np.r_[np.zeros(40, dtype=int), np.ones(40, dtype=int)]
svm_sep = LinearSVC(C=100, max_iter=100000).fit(X_sep, y_sep)
w_sep, b_sep = svm_sep.coef_[0], svm_sep.intercept_[0]


def gap_to_nearest(w, b):
    gap = (np.abs(X_sep @ w + b) / np.linalg.norm(w)).min()
    return gap


midpoint = np.array([1.5, 1.5])
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(cloud_a[:, 0], cloud_a[:, 1], s=22, color=GREY, label="класс −1")
ax.scatter(cloud_b[:, 0], cloud_b[:, 1], s=22, color=BLUE, label="класс +1")
xs = np.linspace(-2, 5, 100)
for name, w, color in [("прямая 1", np.array([0.25, 1.0]), "#7A63B8"), ("прямая 2", np.array([1.0, 0.25]), "#C7B45A")]:
    b = -w @ midpoint
    ax.plot(xs, -(w[0] * xs + b) / w[1], color=color, lw=1.6, label=f"{name}: зазор {gap_to_nearest(w, b):.2f}")
ax.plot(xs, -(w_sep[0] * xs + b_sep) / w_sep[1], color=OCHRE, lw=2.5, label=f"SVM: зазор {gap_to_nearest(w_sep, b_sep):.2f}")
for shift in (-1, 1):
    ax.plot(xs, -(w_sep[0] * xs + b_sep - shift) / w_sep[1], color=OCHRE, lw=1, ls="--")
support_sep = margins_of(X_sep, y_sep, w_sep, b_sep) <= 1 + 1e-6
ax.scatter(X_sep[support_sep, 0], X_sep[support_sep, 1], s=140, facecolor="none", edgecolor=BLACK, lw=1.3, label="опорные векторы")
ax.set_xlim(-2, 5)
ax.set_ylim(-2, 5)
ax.set_aspect("equal")
ax.set_title("Три прямые без ошибок с разным зазором")
ax.legend(loc="upper left", fontsize=9)
plt.show()

Положение прямой метода опорных векторов определяют только объекты на краях
полосы, обведенные черным. Их называют **опорными векторами** (support vectors).

### Постановка задачи

Пара $(w, b)$ задает границу с точностью до множителя: уравнения
$\langle w, x \rangle + b = 0$ и $\langle 2w, x \rangle + 2b = 0$ описывают одну прямую.
Множитель фиксируют условием: у ближайших к границе объектов отступ равен 1.
Тогда края полосы — прямые $\langle w, x \rangle + b = \pm 1$.

Вектор $w$ перпендикулярен границе. Пусть точка $x_1$ лежит на краю
$\langle w, x \rangle + b = 1$; ее сдвигают против направления $w$ на расстояние $d$,
в точку $x_1 - d \frac{w}{\|w\|}$, и подставляют в линейную функцию:

$$\Big\langle w,\ x_1 - d\frac{w}{\|w\|} \Big\rangle + b
= \underbrace{\langle w, x_1 \rangle + b}_{= 1} - d\,\frac{\langle w, w \rangle}{\|w\|}
= 1 - d\,\|w\|.$$

Выражение равно нулю (граница) при $d = \frac{1}{\|w\|}$ и равно $-1$ (противоположный
край) при $d = \frac{2}{\|w\|}$. Ширина полосы:

$$\text{ширина} = \frac{2}{\|w\|}.$$

Наибольшая ширина соответствует наименьшей норме $\|w\|$ при условиях
$M_i \geq 1$ для всех объектов. Если классы неразделимы, условия нарушаются;
за нарушение на объекте $i$ назначают штраф $\max(0, 1 - M_i)$. В записи,
принятой в sklearn:

$$\frac{1}{2}\|w\|^2 + C\sum_{i=1}^{n}\max(0, 1 - M_i) \to \min_{w, b}.$$

Параметр $C$ — цена одного нарушения. Опорными векторами в этой постановке
называют объекты с $M_i \leq 1$: на краю полосы, внутри нее и на чужой стороне.
Только они дают ненулевой вклад в сумму.

### Влияние параметра C

На рисунке граница и полоса при четырех значениях $C$ на данных о винах.
В заголовках — ширина полосы, число опорных векторов и accuracy на тестовом сете.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9.5))
for ax, C in zip(axes.flat, [0.01, 0.1, 1, 100]):
    model = LinearSVC(C=C, max_iter=100000).fit(X_train, y_train)
    w_c, b_c = model.coef_[0], model.intercept_[0]
    support_c = margins_of(X_train, y_train, w_c, b_c) <= 1 + 1e-6
    plot_points(ax, X_train, y_train, alpha=0.35)
    boundary_line(ax, X_train, w_c, b_c, color=OCHRE, band=True)
    ax.scatter(X_train[support_c, 0], X_train[support_c, 1], s=110, facecolor="none", edgecolor=BLACK, lw=1.0)
    ax.set_ylim(-2.2, 3.2)
    ax.set_title(f"C = {C}: ширина {2 / np.linalg.norm(w_c):.2f}, опорных {support_c.sum()}, "
                 f"accuracy {model.score(X_test, y_test):.3f}", fontsize=11)
plt.tight_layout()
plt.show()

При малом $C$ слагаемое $\frac{1}{2}\|w\|^2$ преобладает: норма весов мала,
полоса широкая, почти все объекты опорные. При большом $C$ преобладает сумма
штрафов: полоса сужается, число опорных векторов уменьшается. Значение $C$
выбирают кросс-валидацией.

### Зависимость границы от опорных векторов

Модель обучают заново дважды: только на опорных векторах и на всей выборке
без одного опорного вектора с наименьшим отступом.

In [ ]:
def fit_svm_line(mask):
    model = LinearSVC(C=1, max_iter=100000).fit(X_train[mask], y_train[mask])
    line = (model.coef_[0], model.intercept_[0])
    return line


w_all, b_all = fit_svm_line(np.ones(len(y_train), dtype=bool))
margins_all = margins_of(X_train, y_train, w_all, b_all)
support = margins_all <= 1 + 1e-6
w_support, b_support = fit_svm_line(support)
without_one = np.ones(len(y_train), dtype=bool)
without_one[np.argmin(margins_all)] = False
w_without, b_without = fit_svm_line(without_one)

fig, ax = plt.subplots(figsize=(7, 5.5))
plot_points(ax, X_train, y_train, alpha=0.3)
boundary_line(ax, X_train, w_all, b_all, color=OCHRE, label="вся выборка", lw=6)
boundary_line(ax, X_train, w_support, b_support, color=BLACK, label=f"только {support.sum()} опорных из {len(y_train)}")
boundary_line(ax, X_train, w_without, b_without, color=BLUE, label="без одного опорного вектора")
ax.scatter(*X_train[np.argmin(margins_all)], s=200, facecolor="none", edgecolor=BLUE, lw=2, label="исключенный объект")
ax.set_ylim(-2.2, 3.2)
ax.set_title("Граница зависит только от опорных векторов")
ax.legend(loc="upper right", fontsize=9)
plt.show()

Граница, построенная только по опорным векторам, совпадает с исходной:
у остальных объектов $M_i > 1$, их слагаемые в функции потерь равны нулю.
Исключение одного опорного вектора поворачивает границу. У логистической
регрессии вклад объекта убывает с ростом отступа, но не обращается в ноль,
поэтому на ее границу влияет каждый объект.

## 2. Нелинейная граница: ядра

Набор «полумесяцы» состоит из двух вложенных дуг; линейная граница
их не разделяет.

In [ ]:
X_moons, y_moons = make_moons(n_samples=400, noise=0.2, random_state=SEED)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(X_moons, y_moons, test_size=0.3, random_state=SEED, stratify=y_moons)


def plot_decision(ax, model, X, y, title):
    """Области классов, граница и опорные векторы."""
    grid_x, grid_y = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
                                 np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))
    scores = model.decision_function(np.c_[grid_x.ravel(), grid_y.ravel()]).reshape(grid_x.shape)
    ax.contourf(grid_x, grid_y, scores > 0, levels=1, colors=["#E5E7EB", "#CFE3F7"], alpha=0.9)
    ax.contour(grid_x, grid_y, scores, levels=[0], colors=BLACK, linewidths=1.5)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=14, color=GREY)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=14, color=BLUE)
    if hasattr(model, "support_"):
        ax.scatter(*X[model.support_].T, s=45, facecolor="none", edgecolor=BLACK, lw=0.7)
    ax.set_title(title, fontsize=10)


linear_moons = SVC(kernel="linear", C=1).fit(Xm_train, ym_train)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
plot_decision(ax, linear_moons, Xm_train, ym_train, f"Линейный SVM: accuracy на тестовом сете {linear_moons.score(Xm_test, ym_test):.3f}")
plt.show()

### Новые признаки

К признакам добавляют их произведения и степени до третьей включительно:
$x_1^2, x_1 x_2, x_2^2, x_1^3, \dots$ В новом пространстве линейный SVM
строит гиперплоскость, а в исходных координатах ей соответствует кривая.

In [ ]:
poly_moons = make_pipeline(PolynomialFeatures(degree=3), StandardScaler(), LinearSVC(C=10, max_iter=100000))
poly_moons.fit(Xm_train, ym_train)
n_poly = poly_moons[0].n_output_features_
fig, ax = plt.subplots(figsize=(6.5, 4.5))
plot_decision(ax, poly_moons, Xm_train, ym_train, f"Признаки до третьей степени, {n_poly} шт.: accuracy {poly_moons.score(Xm_test, ym_test):.3f}")
plt.show()

Число одночленов степени не выше $d$ от $p$ переменных равно $\binom{p + d}{d}$.
Для $p = 2$, $d = 3$ это $\binom{5}{3} = 10$, для $p = 784$ пикселей цифры —
$\binom{787}{3} \approx 8 \cdot 10^7$. Ядра дают тот же результат без построения
этих признаков.

### Ядровой переход (kernel trick)

Задача SVM для разделимого случая: $\frac{1}{2}\|w\|^2 \to \min$ при
$y_i(\langle w, x_i \rangle + b) \geq 1$. Условия учитывают множителями Лагранжа (Lagrange multipliers)
$\alpha_i \geq 0$:

$$L(w, b, \alpha) = \frac{1}{2}\|w\|^2 - \sum_{i} \alpha_i \big(y_i(\langle w, x_i \rangle + b) - 1\big).$$

Производные по $w$ и $b$ приравнивают к нулю:

$$\frac{\partial L}{\partial w} = w - \sum_i \alpha_i y_i x_i = 0 \;\Rightarrow\; w = \sum_i \alpha_i y_i x_i,
\qquad \frac{\partial L}{\partial b} = -\sum_i \alpha_i y_i = 0.$$

Вектор весов — линейная комбинация обучающих объектов. В оптимуме $\alpha_i > 0$
только у объектов, для которых условие выполнено как равенство, то есть
у опорных векторов. Подстановка $w$ в модель дает

$$f(x) = \langle w, x \rangle + b = \sum_i \alpha_i y_i \langle x_i, x \rangle + b.$$

Подстановка $w$ в $L$: первое слагаемое $\frac{1}{2}\|w\|^2 = \frac{1}{2}\sum_{i,j}\alpha_i\alpha_j y_i y_j \langle x_i, x_j \rangle$;
сумма $\sum_i \alpha_i y_i \langle w, x_i \rangle$ равна той же двойной сумме без множителя
$\frac{1}{2}$; слагаемое с $b$ равно $b\sum_i \alpha_i y_i = 0$. Остается задача
относительно $\alpha$:

$$\sum_i \alpha_i - \frac{1}{2}\sum_{i,j}\alpha_i \alpha_j y_i y_j \langle x_i, x_j \rangle \to \max_{\alpha \geq 0}.$$

В обучении и в предсказании объекты входят только через скалярные
произведения. Если объекты отобразить в новое пространство $\varphi(x)$,
достаточно значений $K(x, z) = \langle \varphi(x), \varphi(z) \rangle$. Функцию $K$
называют **ядром** (kernel); во многих случаях ее вычисляют без построения $\varphi$.

Пример: $x, z \in \mathbb{R}^2$ и $K(x, z) = \langle x, z \rangle^2$. Раскрытие квадрата дает

$$(x_1 z_1 + x_2 z_2)^2 = x_1^2 z_1^2 + 2 x_1 x_2 z_1 z_2 + x_2^2 z_2^2
= \big\langle (x_1^2, \sqrt{2} x_1 x_2, x_2^2),\ (z_1^2, \sqrt{2} z_1 z_2, z_2^2) \big\rangle,$$

то есть $K$ — скалярное произведение в пространстве признаков
$\varphi(x) = (x_1^2, \sqrt{2} x_1 x_2, x_2^2)$. Равенство проверяют на двух случайных точках.

In [ ]:
x_point, z_point = np.random.default_rng(0).normal(size=(2, 2))


def phi(v):
    features = np.array([v[0] ** 2, np.sqrt(2) * v[0] * v[1], v[1] ** 2])
    return features


print(f"ядро <x, z>^2:              {(x_point @ z_point) ** 2:.6f}")
print(f"скалярное произведение phi: {phi(x_point) @ phi(z_point):.6f}")

В `SVC` чаще всего используют ядра:

- полиномиальное ядро (polynomial kernel) $K(x, z) = (\gamma \langle x, z \rangle + r)^d$, соответствующее всем
  одночленам степени до $d$;
- гауссово ядро, или ядро радиальной базисной функции (radial basis function,
  RBF), $K(x, z) = \exp\big(-\gamma \|x - z\|^2\big)$, равное 1 для совпадающих точек
  и стремящееся к нулю с ростом расстояния; соответствующее пространство
  $\varphi$ бесконечномерно.

Три ядра на одних данных и одном тестовом сете.

In [ ]:
kernel_models = {
    "линейное": SVC(kernel="linear", C=1),
    "полиномиальное, степень 3": SVC(kernel="poly", degree=3, coef0=1, C=1),
    "RBF": SVC(kernel="rbf", gamma=1, C=1),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, (name, model) in zip(axes, kernel_models.items()):
    model.fit(Xm_train, ym_train)
    plot_decision(ax, model, Xm_train, ym_train,
                  f"{name}: accuracy {model.score(Xm_test, ym_test):.3f}, опорных {len(model.support_)}")
plt.tight_layout()
plt.show()

### Параметры gamma и C

Значение RBF-ядра убывает в $e$ раз на расстоянии $\frac{1}{\sqrt{\gamma}}$, поэтому $\gamma$
задает радиус влияния объекта. При малом $\gamma$ влияние распространяется
далеко и граница гладкая; при большом $\gamma$ граница огибает отдельные
объекты. Параметр $C$ имеет тот же смысл, что и в линейном случае.
На рисунке строки соответствуют значениям $C$, столбцы — значениям $\gamma$;
в заголовках accuracy на обучающей выборке и на тестовом сете.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
for row, C in enumerate([0.1, 1, 100]):
    for col, gamma in enumerate([0.1, 1, 30]):
        model = SVC(kernel="rbf", C=C, gamma=gamma).fit(Xm_train, ym_train)
        plot_decision(axes[row, col], model, Xm_train, ym_train,
                      f"C = {C}, gamma = {gamma}: обучение {model.score(Xm_train, ym_train):.2f}, тест {model.score(Xm_test, ym_test):.2f}")
plt.tight_layout()
plt.show()

При $C = 100$, $\gamma = 30$ граница огибает отдельные объекты: accuracy на обучающей
выборке близка к 1, на тестовом сете ниже — модель переобучена. При $C = 0.1$,
$\gamma = 0.1$ граница близка к прямой — модель недообучена. Пару параметров
выбирают кросс-валидацией по сетке значений.

### Масштаб признаков

RBF-ядро зависит от евклидова расстояния, поэтому признак с большим
масштабом определяет расстояние почти целиком. Второй признак набора
умножают на 100 и сравнивают модель без стандартизации и со стандартизацией.

In [ ]:
stretch = np.array([1.0, 100.0])
raw_svm = SVC(kernel="rbf", gamma=1, C=1).fit(Xm_train * stretch, ym_train)
scaled_svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", gamma=1, C=1)).fit(Xm_train * stretch, ym_train)
scales = pd.DataFrame({
    "без стандартизации": [raw_svm.score(Xm_test * stretch, ym_test)],
    "со StandardScaler": [scaled_svm.score(Xm_test * stretch, ym_test)],
}, index=["accuracy на тестовом сете"])
show_table(scales, axis=None)

Без стандартизации расстояние определяется вторым признаком, и accuracy
снижается. Перед SVM с RBF-ядром признаки стандартизуют внутри `Pipeline`.

## 3. Метод главных компонент (PCA)

### Дисперсия проекций

Пусть $X$ — матрица $n \times p$ с центрированными столбцами, $u$ — вектор единичной
длины. **Проекция** (projection) объекта $x_i$ на направление $u$ — число $z_i = \langle u, x_i \rangle$.
Если требуется заменить каждый объект одним числом, выбирают направление,
вдоль которого проекции имеют наибольшую дисперсию: в этом случае сохраняется
наибольшая часть различий между объектами.

На рисунке облако из 300 точек. На первых трех панелях точки спроецированы
на прямые с углами 0°, 90° и на прямую наибольшей дисперсии; проекции
показаны оранжевым. На четвертой панели — дисперсия проекций в зависимости
от угла направления; оранжевые точки соответствуют трем первым панелям.

In [ ]:
cloud = np.random.default_rng(1).multivariate_normal([0, 0], [[3.0, 1.6], [1.6, 1.4]], size=300)
cloud = cloud - cloud.mean(axis=0)
angles = np.deg2rad(np.arange(0, 180))
directions = np.c_[np.cos(angles), np.sin(angles)]
projection_variance = ((cloud @ directions.T) ** 2).mean(axis=0)
best_angle = angles[np.argmax(projection_variance)]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.3), gridspec_kw={"width_ratios": [1, 1, 1, 1.4]})
for ax, angle in zip(axes[:3], [0.0, np.pi / 2, best_angle]):
    u = np.array([np.cos(angle), np.sin(angle)])
    projected = (cloud @ u)[:, None] * u
    ax.scatter(cloud[:, 0], cloud[:, 1], s=6, color=GREY, alpha=0.6)
    ax.plot([-6 * u[0], 6 * u[0]], [-6 * u[1], 6 * u[1]], color=BLACK, lw=1)
    ax.scatter(projected[:, 0], projected[:, 1], s=6, color=OCHRE)
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.set_aspect("equal")
    ax.set_title(f"угол {np.rad2deg(angle):.0f}°: дисперсия {np.var(cloud @ u):.2f}", fontsize=10)
axes[3].plot(np.rad2deg(angles), projection_variance, color=BLUE, lw=2)
for angle in [0.0, np.pi / 2, best_angle]:
    axes[3].scatter(np.rad2deg(angle), np.var(cloud @ np.array([np.cos(angle), np.sin(angle)])), color=OCHRE, s=50, zorder=3)
axes[3].set_xlabel("угол направления, градусы")
axes[3].set_ylabel("дисперсия проекций")
axes[3].set_title("Дисперсия проекций от угла", fontsize=10)
plt.tight_layout()
plt.show()

Дисперсия проекций меняется с углом гладко и имеет один максимум
на полуокружности; ортогональное ему направление дает минимум. Положение
максимума находят аналитически.

### Вывод метода

Дисперсия проекций на направление $u$:

$$\frac{1}{n}\sum_{i=1}^{n} z_i^2 = \frac{1}{n}\sum_{i=1}^{n} u^\top x_i x_i^\top u = u^\top \Big(\frac{1}{n} X^\top X\Big) u = u^\top \Sigma u,$$

где $\Sigma = \frac{1}{n} X^\top X$ — ковариационная матрица (covariance matrix) признаков. Задача:
максимизировать $u^\top \Sigma u$ при условии $u^\top u = 1$. Функция Лагранжа
$u^\top \Sigma u - \lambda (u^\top u - 1)$, ее градиент по $u$:

$$2\Sigma u - 2\lambda u = 0 \;\Rightarrow\; \Sigma u = \lambda u.$$

Оптимальное направление — собственный вектор ковариационной матрицы.
Дисперсия вдоль него $u^\top \Sigma u = u^\top \lambda u = \lambda$, поэтому выбирают собственный
вектор с наибольшим собственным значением; его называют **первой главной
компонентой** (first principal component). Вторая главная компонента — направление наибольшей дисперсии
среди векторов, ортогональных первой, и так далее: главные компоненты —
собственные векторы $\Sigma$, упорядоченные по убыванию собственных значений.

Ковариационная матрица симметрична, поэтому раскладывается как
$\Sigma = V \Lambda V^\top$, где столбцы ортогональной матрицы $V$ — собственные векторы,
$\Lambda$ — диагональная матрица собственных значений. На практике раскладывают
саму матрицу $X$: **сингулярное разложение** (singular value decomposition, SVD)
$X = U S V^\top$, где $U$ и $V$ имеют ортонормированные столбцы, $S$ — диагональная
матрица сингулярных чисел $s_1 \geq s_2 \geq \dots \geq 0$. Из него

$$X^\top X = V S U^\top U S V^\top = V S^2 V^\top \;\Rightarrow\; \Sigma = V \, \frac{S^2}{n} \, V^\top,$$

то есть матрица $V$ в обоих разложениях одна и та же, а $\lambda_j = \frac{s_j^2}{n}$.

### Разложения на примере

Пример: 8 объектов, 5 признаков. Сначала разложение ковариационной матрицы
$\Sigma = V \Lambda V^\top$, затем сингулярное разложение центрированной матрицы
$X = U S V^\top$. В клетках — значения элементов.

In [ ]:
def show_product(matrices, titles, symbols, figsize):
    """Матрицы как тепловые карты с подписями и знаками между ними."""
    widths = []
    for k, matrix in enumerate(matrices):
        widths.append(matrix.shape[1])
        if k < len(symbols):
            widths.append(0.8)
    fig, axes = plt.subplots(1, len(widths), figsize=figsize, gridspec_kw={"width_ratios": widths})
    limit = max(np.abs(m).max() for m in matrices)
    max_rows = max(m.shape[0] for m in matrices)
    position = 0
    for k, (matrix, title) in enumerate(zip(matrices, titles)):
        ax = axes[position]
        rows, cols = matrix.shape
        pad = (max_rows - rows) / 2
        ax.imshow(matrix, cmap="RdBu", vmin=-limit, vmax=limit)
        for i in range(rows):
            for j in range(cols):
                ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=7)
        ax.add_patch(Rectangle((-0.5, -0.5), cols, rows, fill=False, lw=1.2, color=BLACK))
        ax.text((cols - 1) / 2, -0.8, title, ha="center", va="bottom", fontsize=10)
        ax.set_xlim(-0.5, cols - 0.5)
        ax.set_ylim(rows - 0.5 + pad, -0.5 - pad - 0.9)
        ax.axis("off")
        position += 1
        if k < len(symbols):
            axes[position].axis("off")
            axes[position].text(0.5, 0.5, symbols[k], ha="center", va="center", fontsize=28)
            position += 1
    plt.tight_layout()
    plt.show()


X_small = np.random.default_rng(3).normal(size=(8, 5)) @ np.random.default_rng(4).normal(size=(5, 5))
X_small = X_small - X_small.mean(axis=0)
sigma_small = X_small.T @ X_small / len(X_small)
eig_values, eig_vectors = np.linalg.eigh(sigma_small)
order = np.argsort(eig_values)[::-1]
eig_values, eig_vectors = eig_values[order], eig_vectors[:, order]
show_product([sigma_small, eig_vectors, np.diag(eig_values), eig_vectors.T],
             ["Σ, 5 × 5", "V: собственные векторы по столбцам", "Λ: собственные значения", "Vᵀ"],
             ["=", "·", "·"], figsize=(16, 3.4))

In [ ]:
U_small, s_small, Vt_small = np.linalg.svd(X_small, full_matrices=False)
show_product([X_small, U_small, np.diag(s_small), Vt_small],
             ["X, 8 × 5", "U, 8 × 5", "S: сингулярные числа", "Vᵀ, 5 × 5"],
             ["=", "·", "·"], figsize=(16, 4.2))
print("собственные значения Σ:", np.round(eig_values, 3))
print("s² / n:                ", np.round(s_small ** 2 / len(X_small), 3))
print("|<строка Vᵀ из SVD, столбец V из Σ>|:", np.round(np.abs((Vt_small * eig_vectors.T).sum(axis=1)), 3))

Строки $V^\top$ из сингулярного разложения совпадают со столбцами $V$ из разложения
ковариационной матрицы с точностью до знака, $\frac{s_j^2}{n}$ совпадают с собственными
значениями. Знак компоненты не определен: векторы $u$ и $-u$ задают одно
направление.

### Снижение размерности через сингулярное разложение

Пусть $r = \min(n, p)$. Оставляют первые $k < r$ слагаемых разложения: первые $k$
столбцов $U$ (матрица $U_k$ размера $n \times k$), левый верхний блок $S_k$ размера
$k \times k$ и первые $k$ строк $V^\top$ (матрица $V_k^\top$ размера $k \times p$). Получают
матрицу ранга $k$:

$$X_k = U_k S_k V_k^\top = (U_k S_k)\, V_k^\top.$$

Схема показывает, какие блоки сомножителей входят в $X_k$.

In [ ]:
def draw_matrix(ax, x, y, width, height, name, shape, keep=None, keep_label=None):
    """Прямоугольник матрицы; keep = ('cols' | 'rows' | 'block', доля) выделяет сохраняемую часть."""
    ax.add_patch(Rectangle((x, y), width, height, fill=False, lw=2, color=BLACK))
    ax.text(x + width / 2, y + height + 0.55, name, ha="center", fontsize=15)
    ax.text(x + width / 2, y + height + 0.2, shape, ha="center", fontsize=11, color=GREY)
    if keep is None:
        ax.text(x + width / 2, y + height / 2, keep_label, ha="center", va="center", fontsize=13)
        return
    kind, share = keep
    if kind == "cols":
        part = (x, y, width * share, height)
        ax.text(x + width * share / 2, y - 0.35, "k", ha="center", fontsize=12)
    elif kind == "rows":
        part = (x, y + height * (1 - share), width, height * share)
        ax.text(x + width + 0.15, y + height * (1 - share / 2), "k", va="center", fontsize=12)
    else:
        part = (x, y + height * (1 - share), width * share, height * share)
        ax.text(x + width * share + 0.15, y + height * (1 - share / 2), "k", va="center", fontsize=12)
        ax.text(x + width * share / 2, y + height * (1 - share) - 0.35, "k", ha="center", fontsize=12)
    ax.add_patch(Rectangle(part[:2], part[2], part[3], color=OCHRE, alpha=0.35, lw=0))
    ax.add_patch(Rectangle(part[:2], part[2], part[3], fill=False, lw=1.5, color=BLACK))
    ax.text(part[0] + part[2] / 2, part[1] + part[3] / 2, keep_label, ha="center", va="center", fontsize=13)


fig, ax = plt.subplots(figsize=(13, 5))
draw_matrix(ax, 0.0, 0, 2.4, 4, "X", "n × p", keep_label="Xₖ")
ax.text(3.0, 2.8, "≈", fontsize=26, ha="center", va="center")
draw_matrix(ax, 3.6, 0, 2.4, 4, "U", "n × r", keep=("cols", 0.4), keep_label="Uₖ")
ax.text(6.6, 2.8, "·", fontsize=26, ha="center", va="center")
draw_matrix(ax, 7.2, 1.6, 2.4, 2.4, "S", "r × r", keep=("block", 0.4), keep_label="Sₖ")
ax.text(10.2, 2.8, "·", fontsize=26, ha="center", va="center")
draw_matrix(ax, 10.8, 1.6, 2.4, 2.4, "Vᵀ", "r × p", keep=("rows", 0.4), keep_label="Vₖᵀ")
ax.set_xlim(-0.3, 13.8)
ax.set_ylim(-0.8, 5.2)
ax.set_aspect("equal")
ax.axis("off")
plt.show()

Смысл сомножителей устанавливают умножением разложения справа на $V$.
Так как $V^\top V = I$,

$$X V = U S V^\top V = U S,$$

а для первых $k$ столбцов $X V_k = U_k S_k$. Строка $i$ матрицы $X V_k$ содержит скалярные
произведения объекта $x_i$ с первыми $k$ главными компонентами, то есть его
проекции. Отсюда:

- $Z = U_k S_k = X V_k$ — матрица $n \times k$ новых координат объектов, результат `transform`;
- $V_k^\top$ — матрица $k \times p$, строки которой — главные компоненты, атрибут `components_`;
- $X_k = Z V_k^\top$ — восстановление объектов по $k$ координатам, результат `inverse_transform`
  без прибавления среднего.

Ошибка приближения равна сумме квадратов отброшенных сингулярных чисел:
$\|X - X_k\|_F^2 = \sum_{j > k} s_j^2$, так как $X - X_k = \sum_{j > k} s_j u_j v_j^\top$, а слагаемые
ортогональны. По теореме Эккарта — Янга (Eckart — Young theorem) никакая матрица ранга $k$ не дает
меньшей ошибки. Для матрицы примера при $k = 2$:

In [ ]:
k_small = 2
Z_small = U_small[:, :k_small] * s_small[:k_small]
X_small_k = Z_small @ Vt_small[:k_small]
show_product([Z_small, Vt_small[:k_small], X_small_k],
             ["Z = UₖSₖ, 8 × 2", "Vₖᵀ, 2 × 5", "Xₖ, 8 × 5"],
             ["·", "="], figsize=(8, 4.2))
print(f"||X − X_k||² = {((X_small - X_small_k) ** 2).sum():.4f}")
print(f"Σ s_j² при j > k = {(s_small[k_small:] ** 2).sum():.4f}")
print("X V_k совпадает с U_k S_k:", np.allclose(X_small @ Vt_small[:k_small].T, Z_small))

Тот же результат дает класс `PCA` из sklearn. На облаке точек сравнивают три
способа; sklearn делит на $n - 1$, а не на $n$, поэтому его дисперсии умножены
на $\frac{n - 1}{n}$. Совпадение направлений проверяют модулем косинуса угла
между компонентами: для единичных векторов он равен $|\langle u, v \rangle|$.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(cloud.T @ cloud / len(cloud))
by_eig = eigenvectors[:, ::-1].T
_, singular, vt = np.linalg.svd(cloud, full_matrices=False)
sk_pca = PCA(n_components=2).fit(cloud)

pca_check = pd.DataFrame({
    "собственные значения Σ": eigenvalues[::-1],
    "SVD: s² / n": singular ** 2 / len(cloud),
    "sklearn: explained_variance_ · (n−1)/n": sk_pca.explained_variance_ * (len(cloud) - 1) / len(cloud),
    "косинус угла: SVD и sklearn": np.abs((vt * sk_pca.components_).sum(axis=1)),
    "косинус угла: SVD и собственный вектор": np.abs((vt * by_eig).sum(axis=1)),
}, index=["компонента 1", "компонента 2"])
show_table(pca_check, digits=4)

### Компоненты и объясненная дисперсия

Для набора объектов с центром $\mu$ используют следующие величины:

- **главная компонента** (principal component) $u_j \in \mathbb{R}^p$ — единичный вектор, $j$-й собственный
  вектор ковариационной матрицы;
- **координата** (score) объекта по компоненте: $z_j = \langle u_j, x - \mu \rangle$;
- **дисперсия вдоль компоненты** $\lambda_j$ — выборочная дисперсия координат $z_j$,
  она равна $j$-му собственному значению;
- **доля объясненной дисперсии** (explained variance ratio) $\frac{\lambda_j}{\sum_{l=1}^{p} \lambda_l}$.

Знаменатель равен суммарной дисперсии всех исходных признаков. Действительно,
$\sum_l \lambda_l = \mathrm{tr}\,\Lambda = \mathrm{tr}(V^\top \Sigma V) = \mathrm{tr}(\Sigma V V^\top) = \mathrm{tr}\,\Sigma = \sum_{m=1}^{p} \mathrm{Var}(x_m)$:
переход к компонентам не меняет общую дисперсию, а перераспределяет ее так,
что первые компоненты получают наибольшую долю.

### Набор MNIST

Набор содержит 70 000 изображений рукописных цифр размером 28 × 28 пикселей;
каждое изображение — вектор из 784 яркостей от 0 до 1. Используют 12 000
изображений: 10 000 для обучения и 2000 для тестового сета.

In [ ]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X_digits_all = mnist.data.astype(np.float32) / 255
y_digits_all = mnist.target.astype(int)
X_digits, _, y_digits, _ = train_test_split(X_digits_all, y_digits_all, train_size=12000, random_state=SEED, stratify=y_digits_all)
Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_digits, y_digits, test_size=2000, random_state=SEED, stratify=y_digits)
print("обучение:", Xd_train.shape, " тестовый сет:", Xd_test.shape)

In [ ]:
def show_digits(images, titles, n_cols=10, size=1.3):
    n_rows = int(np.ceil(len(images) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(size * n_cols, size * 1.2 * n_rows))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, image, title in zip(np.atleast_1d(axes).ravel(), images, titles):
        ax.imshow(np.asarray(image).reshape(28, 28), cmap="gray_r")
        ax.set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()


show_digits(Xd_train[:20], [str(label) for label in yd_train[:20]])

Метод `fit` класса `PCA` не подбирает параметры по ошибке, как при обучении
модели. Он вычисляет вектор средних $\mu$, сингулярное разложение центрированной
матрицы и по нему компоненты и дисперсии. Эти величины вычисляют
по обучающей выборке; тестовый сет в вычислении не участвует, как
и при стандартизации, иначе часть информации о нем попадает в преобразование.

Вычисляют все 784 компоненты и строят накопленную долю объясненной
дисперсии $\frac{\lambda_1 + \dots + \lambda_k}{\lambda_1 + \dots + \lambda_{784}}$ в зависимости от $k$.

In [ ]:
pca_digits = PCA().fit(Xd_train)
cumulative = np.cumsum(pca_digits.explained_variance_ratio_)
needed = {share: int(np.searchsorted(cumulative, share) + 1) for share in [0.8, 0.9, 0.95]}

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, 785), cumulative, color=BLUE, lw=2)
for (share, k), color in zip(needed.items(), [GREY, OCHRE, BLACK]):
    plt.axvline(k, color=color, ls="--", lw=1, label=f"{share:.0%} дисперсии: {k} компонент")
plt.xscale("log")
plt.xlabel("число компонент k")
plt.ylabel("накопленная доля дисперсии")
plt.legend()
plt.show()

Девяносто процентов суммарной дисперсии приходится менее чем на сто
компонент из 784: яркости соседних пикселей сильно коррелированы, а пиксели
у краев изображения почти всегда равны нулю и имеют малую дисперсию.

Каждая компонента $u_j$ — вектор из 784 чисел, его можно изобразить
как картинку 28 × 28. На рисунке вектор средних и первые восемь компонент;
синий цвет — положительные элементы, красный — отрицательные.

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(14, 2))
axes[0].imshow(pca_digits.mean_.reshape(28, 28), cmap="gray_r")
axes[0].set_title("среднее μ", fontsize=9)
for j, ax in enumerate(axes[1:]):
    component = pca_digits.components_[j].reshape(28, 28)
    limit = np.abs(component).max()
    ax.imshow(component, cmap="RdBu", vmin=-limit, vmax=limit)
    ax.set_title(f"u{j + 1}, λ = {pca_digits.explained_variance_[j]:.1f}", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Разложение изображения по компонентам

Компоненты образуют ортонормированный базис пространства $\mathbb{R}^{784}$, поэтому
любое изображение записывается точно:

$$x = \mu + \sum_{j=1}^{784} z_j u_j, \qquad z_j = \langle u_j, x - \mu \rangle.$$

Если оставить только первые $k$ слагаемых, получают приближение
$\hat{x}_k = \mu + \sum_{j=1}^{k} z_j u_j$ — строку матрицы $X_k$ из предыдущего раздела
с прибавленным средним. Средняя по выборке квадратичная ошибка приближения
равна сумме отброшенных дисперсий $\lambda_{k+1} + \dots + \lambda_{784}$, и по теореме
Эккарта — Янга первые $k$ компонент дают наименьшую ошибку среди всех
$k$-мерных приближений.

### В чем состоит снижение размерности

Изображения $u_j$ имеют тот же размер 28 × 28, что и исходное изображение, так как
это векторы того же пространства $\mathbb{R}^{784}$. Сжатие состоит в другом: вектор
средних $\mu$ и компоненты $u_1, \dots, u_k$ вычисляют один раз по обучающей выборке,
и они общие для всех изображений. Каждое отдельное изображение описывают
только вектором координат $z = (z_1, \dots, z_k)$ — $k$ числами вместо 784.

Для всей выборки разложение записывают в матричной форме. Пусть строки
матрицы $X$ размера $n \times 784$ — изображения, $V_k$ — матрица $784 \times k$
со столбцами $u_1, \dots, u_k$. Координаты всех изображений
$Z = (X - \mathbf{1}\mu^\top) V_k$ — матрица $n \times k$, и

$$\underbrace{X - \mathbf{1}\mu^\top}_{n \times 784} \approx \underbrace{Z}_{n \times k} \cdot \underbrace{V_k^\top}_{k \times 784}.$$

Это разложение $X_k = U_k S_k V_k^\top$ из раздела о сингулярном разложении с $Z = U_k S_k$.
Строка $i$ матрицы $Z$ — координаты изображения $i$; строки $V_k^\top$ — компоненты,
одни для всех строк $Z$. На схеме выделены одно изображение и его координаты.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
row_y, row_h = 2.6, 0.35
ax.add_patch(Rectangle((0, 0), 4.8, 4, fill=False, lw=2, color=BLACK))
ax.add_patch(Rectangle((0, row_y), 4.8, row_h, color=OCHRE, alpha=0.45, lw=0))
ax.text(2.4, row_y + row_h / 2, "изображение i: 784 яркости", ha="center", va="center", fontsize=11)
ax.text(2.4, 4.55, "X − μ", ha="center", fontsize=15)
ax.text(2.4, 4.2, "n × 784", ha="center", fontsize=11, color=GREY)
ax.text(5.4, 2, "≈", fontsize=26, ha="center", va="center")

ax.add_patch(Rectangle((6.0, 0), 0.9, 4, fill=False, lw=2, color=BLACK))
ax.add_patch(Rectangle((6.0, row_y), 0.9, row_h, color=OCHRE, alpha=0.45, lw=0))
ax.text(6.45, row_y + row_h / 2, "zᵢ", ha="center", va="center", fontsize=12)
ax.text(6.45, 4.55, "Z", ha="center", fontsize=15)
ax.text(6.45, 4.2, "n × k", ha="center", fontsize=11, color=GREY)
ax.text(6.45, -0.4, "k чисел\nна изображение", ha="center", va="top", fontsize=10)
ax.text(7.5, 2, "·", fontsize=30, ha="center", va="center")

strip_names = ["u₁ᵀ", "u₂ᵀ", "…", "uₖᵀ"]
for m, name in enumerate(strip_names):
    y = 3.1 - m * 0.3
    ax.add_patch(Rectangle((8.1, y), 4.8, 0.3, color=BLUE, alpha=0.12 + 0.08 * (m % 2), lw=0))
    ax.text(10.5, y + 0.15, name, ha="center", va="center", fontsize=11)
ax.add_patch(Rectangle((8.1, 2.2), 4.8, 1.2, fill=False, lw=2, color=BLACK))
ax.text(10.5, 4.55, "Vₖᵀ", ha="center", fontsize=15)
ax.text(10.5, 4.2, "k × 784", ha="center", fontsize=11, color=GREY)
ax.text(10.5, 1.9, "компоненты: вычисляют один раз,\nобщие для всех изображений", ha="center", va="top", fontsize=10)
ax.set_xlim(-0.3, 13.2)
ax.set_ylim(-1.2, 5.0)
ax.set_aspect("equal")
ax.axis("off")
plt.show()

Для одного изображения из тестового сета при $k = 50$: метод `transform`
вычисляет $z = V_k^\top (x - \mu)$ — 50 чисел, метод `inverse_transform`
восстанавливает $\hat{x} = \mu + V_k z$ — снова 784 яркости.

In [ ]:
digit = Xd_test[0]
coordinates = pca_digits.transform(digit[None, :])[0]
k_code = 50
digit_restored = pca_digits.mean_ + coordinates[:k_code] @ pca_digits.components_[:k_code]

fig, axes = plt.subplots(1, 5, figsize=(16, 3.6), gridspec_kw={"width_ratios": [1, 0.55, 2.6, 0.7, 1]})
axes[0].imshow(digit.reshape(28, 28), cmap="gray_r")
axes[0].set_title("x: 784 яркости", fontsize=11)
axes[2].bar(np.arange(1, k_code + 1), coordinates[:k_code], color=BLUE)
axes[2].axhline(0, color=BLACK, lw=0.8)
axes[2].set_xlabel("номер компоненты j")
axes[2].set_ylabel("координата zⱼ")
axes[2].set_title(f"z = Vₖᵀ(x − μ): {k_code} чисел", fontsize=11)
axes[4].imshow(digit_restored.reshape(28, 28), cmap="gray_r")
axes[4].set_title("x̂ = μ + Vₖz: 784 яркости", fontsize=11)
for ax, label in [(axes[1], "transform"), (axes[3], "inverse_transform")]:
    ax.annotate("", xy=(0.95, 0.5), xytext=(0.05, 0.5), arrowprops={"arrowstyle": "->", "lw": 2, "color": BLACK})
    ax.text(0.5, 0.6, label, ha="center", va="bottom", fontsize=11)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
for ax in (axes[0], axes[1], axes[3], axes[4]):
    ax.axis("off")
plt.tight_layout()
plt.show()

Объем хранения выборки из $n$ изображений без сжатия — $784\,n$ чисел, после сжатия —
$k\,n$ координат и $784\,k + 784$ чисел компонент и среднего, которые не зависят от $n$.
В таблице строки — способы хранения обучающей выборки, столбцы — число
хранимых чисел.

In [ ]:
n_images = len(Xd_train)
storage = pd.DataFrame({
    "чисел на изображение": [784, k_code],
    "общих чисел (μ и компоненты)": [0, 784 * k_code + 784],
    "чисел всего": [784 * n_images, k_code * n_images + 784 * k_code + 784],
}, index=["исходные изображения", f"PCA, k = {k_code}"])
print(f"сжатие в {storage['чисел всего'].iloc[0] / storage['чисел всего'].iloc[1]:.1f} раза")
show_table(storage, digits=0, lower_is_better=True)

Общая часть хранится один раз, поэтому при большом $n$ коэффициент сжатия
стремится к $\frac{784}{k}$. Кроме хранения, сокращается число признаков
для последующих моделей: метод k ближайших соседей (k nearest neighbors) ниже вычисляет
расстояния между векторами $z$ из 50 чисел вместо 784.

### Изображение как сумма компонент

Восстановление $\hat{x} = \mu + V_k z$ — это сумма $\mu + z_1 u_1 + \dots + z_k u_k$.
На схеме изображение из тестового сета записано как сумма вектора средних
и первых пяти компонент с коэффициентами $z_j$.

In [ ]:
n_terms = 5
parts = [(digit, "x", "gray_r"), ("=", None, None), (pca_digits.mean_, "μ", "gray_r")]
for j in range(n_terms):
    sign = "+" if coordinates[j] >= 0 else "−"
    parts += [(f"{sign} {abs(coordinates[j]):.1f} ·", None, None), (pca_digits.components_[j], f"u{j + 1}", "RdBu")]
parts += [("+ …", None, None)]

fig, axes = plt.subplots(1, len(parts), figsize=(18, 2.6), gridspec_kw={"width_ratios": [3, 1] * (len(parts) // 2)})
for ax, (content, title, cmap) in zip(axes, parts):
    ax.axis("off")
    if isinstance(content, str):
        ax.text(0.5, 0.5, content, ha="center", va="center", fontsize=13)
        continue
    if cmap == "RdBu":
        limit = np.abs(content).max()
        ax.imshow(content.reshape(28, 28), cmap=cmap, vmin=-limit, vmax=limit)
    else:
        ax.imshow(content.reshape(28, 28), cmap=cmap)
    ax.set_title(title, fontsize=11)
plt.show()

Частичные суммы $\hat{x}_k$ при росте $k$ приближаются к исходному изображению.
Для того же изображения — приближения по 2, 10, 30, 100 и 300 компонентам.

In [ ]:
restored = [pca_digits.mean_ + coordinates[:k] @ pca_digits.components_[:k] for k in [2, 10, 30, 100, 300]]
show_digits([digit] + restored, ["исходное"] + [f"k = {k}" for k in [2, 10, 30, 100, 300]], n_cols=6, size=1.8)

Средняя по тестовому сету квадратичная ошибка восстановления в зависимости
от числа компонент.

In [ ]:
test_coordinates = pca_digits.transform(Xd_test)
k_grid = [1, 2, 5, 10, 20, 50, 100, 200, 400, 784]
errors = []
for k in k_grid:
    restored_test = pca_digits.mean_ + test_coordinates[:, :k] @ pca_digits.components_[:k]
    errors.append(((Xd_test - restored_test) ** 2).sum(axis=1).mean())
plt.figure(figsize=(8, 3.6))
plt.plot(k_grid, errors, "o-", color=BLUE)
plt.xscale("log")
plt.xlabel("число компонент k")
plt.ylabel("средняя ошибка ‖x − x̂‖²")
plt.title("Ошибка восстановления на тестовом сете")
plt.show()

### Изображения на плоскости первых двух компонент

Каждое изображение обучающей выборки заменяют парой координат $(z_1, z_2)$.
На статичном рисунке точки окрашены по цифре, поверх отдельных точек
показаны сами изображения в рамке цвета их цифры. Интерактивный рисунок
ниже показывает цифру и номер изображения при наведении курсора; щелчок
по цифре в легенде скрывает или показывает ее точки.

In [ ]:
def spread_indices(points, n_marks, min_share=0.07, seed=0):
    """Номера точек, разнесенных друг от друга не меньше чем на min_share размаха."""
    span = points.max(axis=0) - points.min(axis=0)
    chosen = []
    for index in np.random.default_rng(seed).permutation(len(points)):
        if all(np.linalg.norm((points[index] - points[other]) / span) > min_share for other in chosen):
            chosen.append(index)
        if len(chosen) == n_marks:
            break
    return chosen


def plot_digit_map(ax, points, labels, images, title, n_marks=40):
    for digit_class in range(10):
        mask = labels == digit_class
        ax.scatter(points[mask, 0], points[mask, 1], s=4, color=CLUSTER_COLORS[digit_class], alpha=0.5)
    for index in spread_indices(points, n_marks):
        thumb = OffsetImage(images[index].reshape(28, 28), cmap="gray_r", zoom=0.55)
        box = AnnotationBbox(thumb, points[index], frameon=True, pad=0.1,
                             bboxprops={"edgecolor": CLUSTER_COLORS[labels[index]], "lw": 1.5})
        ax.add_artist(box)
    handles = [Line2D([], [], marker="o", ls="", color=CLUSTER_COLORS[d], label=str(d)) for d in range(10)]
    ax.legend(handles=handles, title="цифра", loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])


def interactive_digit_map(points, labels, title, axis_names=("ось 1", "ось 2")):
    frame = pd.DataFrame({axis_names[0]: points[:, 0], axis_names[1]: points[:, 1],
                          "цифра": labels.astype(str), "номер изображения": np.arange(len(labels))})
    figure = px.scatter(frame, x=axis_names[0], y=axis_names[1], color="цифра",
                        hover_data={"цифра": True, "номер изображения": True},
                        category_orders={"цифра": [str(d) for d in range(10)]},
                        color_discrete_sequence=list(CLUSTER_COLORS), opacity=0.7,
                        width=850, height=650, title=title)
    figure.update_traces(marker_size=5)
    figure.show()


plane = pca_digits.transform(Xd_train[:3000])[:, :2]
labels_plane = yd_train[:3000]
fig, ax = plt.subplots(figsize=(11, 9))
plot_digit_map(ax, plane, labels_plane, Xd_train[:3000],
               f"MNIST на плоскости первых двух главных компонент: {cumulative[1]:.0%} дисперсии")
ax.set_xlabel("z₁")
ax.set_ylabel("z₂")
plt.show()

In [ ]:
interactive_digit_map(plane, labels_plane, "MNIST на плоскости первых двух главных компонент", ("z₁", "z₂"))

Вдоль первой компоненты отделяются нули и единицы, остальные цифры
перекрываются: две компоненты из 784 сохраняют лишь малую долю дисперсии,
указанную в заголовке рисунка.

### PCA перед классификатором

Метод k ближайших соседей (k nearest neighbors, kNN) вычисляет расстояния
в 784-мерном пространстве. Изображения сжимают до $k$ компонент и сравнивают
accuracy и время предсказания на одном тестовом сете. PCA входит в `Pipeline`
и вычисляется только по обучающей выборке.

In [ ]:
pca_rows = {}
for k in [5, 10, 20, 50, 100, 200, 784]:
    steps = [KNeighborsClassifier(n_neighbors=5)] if k == 784 else [PCA(n_components=k, random_state=SEED), KNeighborsClassifier(n_neighbors=5)]
    model = make_pipeline(*steps).fit(Xd_train, yd_train)
    start = time.perf_counter()
    accuracy = model.score(Xd_test, yd_test)
    pca_rows[k] = {"accuracy": accuracy, "время предсказания, с": time.perf_counter() - start}
pca_table = pd.DataFrame(pca_rows).T.rename_axis("компонент")
show_table(pca_table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(pca_table.index, pca_table["accuracy"], "o-", color=BLUE)
axes[0].set_ylabel("accuracy на тестовом сете")
axes[1].plot(pca_table.index, pca_table["время предсказания, с"], "o-", color=OCHRE)
axes[1].set_ylabel("время предсказания, с")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("число компонент; 784 — без PCA")
plt.tight_layout()
plt.show()

При 50 компонентах accuracy не ниже, чем на исходных пикселях, а предсказание
выполняется быстрее. Отброшенные компоненты имеют малую дисперсию
и описывают в основном индивидуальные особенности почерка; без них
расстояния между изображениями лучше отражают различие цифр. При пяти
компонентах сохраняется слишком малая доля дисперсии, и accuracy падает.

## 4. Визуализация многомерных данных

PCA — линейное преобразование, сохраняющее наибольшую дисперсию. Для изображения
данных на плоскости часто важнее другое свойство: точки, близкие в исходном
пространстве, должны остаться близкими на плоскости. Методы t-SNE и UMAP
строят двумерное представление, оптимизируя именно это свойство. Взаимное
расположение далеких точек при этом сохраняется плохо.

Оба метода применяют к 2500 изображениям обучающей выборки, предварительно
сжатым PCA до 50 компонент. Сжатие уменьшает время вычисления расстояний
и удаляет компоненты с малой дисперсией.

In [ ]:
vis_X, vis_y = Xd_train[:2500], yd_train[:2500]
vis_pca50 = PCA(n_components=50, random_state=SEED).fit_transform(vis_X)
print("объектов:", vis_pca50.shape[0], " признаков после PCA:", vis_pca50.shape[1])

### Стохастическое вложение соседей с распределением Стьюдента (t-SNE)

Пусть $n$ — число объектов выборки, $x_1, \dots, x_n$ — объекты в исходном пространстве,
$y_1, \dots, y_n$ — их образы на плоскости.

**Сходство в исходном пространстве.** Для объекта $x_i$ задают распределение
на остальных объектах — вероятность выбрать $x_j$ соседом:

$$p_{j \mid i} = \frac{\exp\big(-\|x_i - x_j\|^2 / 2\sigma_i^2\big)}{\sum_{l \neq i} \exp\big(-\|x_i - x_l\|^2 / 2\sigma_i^2\big)}, \qquad p_{i \mid i} = 0.$$

Сходство пары симметризуют и нормируют на всю выборку:

$$p_{ij} = \frac{p_{j \mid i} + p_{i \mid j}}{2n}, \qquad \sum_{i \neq j} p_{ij} = 1.$$

Делитель $2n$ обеспечивает нормировку: каждое из $n$ распределений $P_i$ дает в сумму
единицу, и каждая пара учтена дважды.

**Перплексия.** Ширину $\sigma_i$ подбирают для каждого объекта отдельно. Энтропия (entropy)
распределения $P_i = (p_{j \mid i})_j$:

$$H(P_i) = -\sum_{j} p_{j \mid i} \log_2 p_{j \mid i},$$

**перплексия** (perplexity):

$$\mathrm{Perp}(P_i) = 2^{H(P_i)}.$$

Если распределение равномерно на $k$ объектах, $p_{j \mid i} = \frac{1}{k}$, и подстановка дает

$$H = -\sum_{j=1}^{k} \frac{1}{k} \log_2 \frac{1}{k} = \log_2 k, \qquad \mathrm{Perp} = 2^{\log_2 k} = k.$$

Поэтому перплексию трактуют как эффективное число соседей: распределение
с перплексией $k$ имеет ту же неопределенность, что и равномерный выбор
из $k$ вариантов. Термин (от лат. perplexus — запутанный) пришел из теории
информации и языкового моделирования, где перплексия модели — эффективное
число равновероятных вариантов следующего слова. Аналогичная величина
в статистической физике — число равновероятных микросостояний $W$ в формуле
Больцмана $S = k_B \ln W$: системе с энтропией $S$ соответствует $W = e^{S / k_B}$
равновероятных состояний, так же как распределению с энтропией $H$ бит
соответствует $2^H$ равновероятных исходов.

Параметр `perplexity` метода t-SNE задает требуемое значение: для каждого $i$
значение $\sigma_i$ находят двоичным поиском из уравнения
$\mathrm{Perp}(P_i) = $ `perplexity`. Перплексия возрастает с ростом $\sigma_i$, так как
распределение становится ближе к равномерному, поэтому решение единственно.
В плотной области соседи близко, и нужное число соседей набирается при малом
$\sigma_i$; в разреженной области $\sigma_i$ больше. На схеме две точки в областях
разной плотности при `perplexity` = 10; площадь точек пропорциональна $p_{j \mid i}$,
окружность имеет радиус $\sigma_i$.

In [ ]:
def conditional_p(X, i, perplexity):
    """p_{j|i} и sigma_i, подобранная двоичным поиском под заданную перплексию."""
    squared = ((X - X[i]) ** 2).sum(axis=1)
    low, high = 1e-3, 1e3
    for _ in range(100):
        sigma = np.sqrt(low * high)
        weights = np.exp(-squared / (2 * sigma ** 2))
        weights[i] = 0
        p = weights / weights.sum()
        entropy = -(p[p > 0] * np.log2(p[p > 0])).sum()
        if 2 ** entropy > perplexity:
            high = sigma
        else:
            low = sigma
    return p, sigma


rng_density = np.random.default_rng(5)
toy_points = np.vstack([rng_density.normal([0, 0], 0.3, (60, 2)), rng_density.normal([4, 0], 1.2, (60, 2))])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, i, title in zip(axes, [0, 60], ["точка в плотной области", "точка в разреженной области"]):
    p, sigma = conditional_p(toy_points, i, perplexity=10)
    views = [(ax, 8 + 1500 * p)]
    if i == 0:
        zoom = ax.inset_axes([0.42, 0.55, 0.42, 0.42])
        zoom.set_xlim(toy_points[i, 0] - 0.6, toy_points[i, 0] + 0.6)
        zoom.set_ylim(toy_points[i, 1] - 0.6, toy_points[i, 1] + 0.6)
        zoom.set_xticks([])
        zoom.set_yticks([])
        zoom.set_title("увеличенный фрагмент", fontsize=9)
        ax.indicate_inset_zoom(zoom, edgecolor=GREY)
        views.append((zoom, 8 + 1500 * p))
    for view, sizes in views:
        view.scatter(toy_points[:, 0], toy_points[:, 1], s=sizes, color=BLUE, alpha=0.5)
        view.scatter(*toy_points[i], s=160, color=OCHRE, marker="*", zorder=3)
        view.add_patch(Circle(toy_points[i], sigma, fill=False, color=OCHRE, lw=1.5))
        view.set_aspect("equal")
    ax.set_xlim(-2, 8)
    ax.set_ylim(-4, 4)
    ax.set_title(f"{title}: σ = {sigma:.2f}, perplexity = 10", fontsize=10)
plt.tight_layout()
plt.show()

**Сходство на плоскости** задают распределением Стьюдента (Student's t-distribution) с одной степенью свободы:

$$q_{ij} = \frac{\big(1 + \|y_i - y_j\|^2\big)^{-1}}{\sum_{l \neq m} \big(1 + \|y_l - y_m\|^2\big)^{-1}}.$$

**Функция потерь** — расхождение Кульбака — Лейблера (Kullback — Leibler, KL)
между распределениями $P$ и $Q$:

$$\mathrm{KL}(P \,\|\, Q) = \sum_{i \neq j} p_{ij} \ln \frac{p_{ij}}{q_{ij}}.$$

Наибольший вклад дают пары с большим $p_{ij}$ и малым $q_{ij}$, то есть близкие в исходном
пространстве объекты, образы которых далеки на плоскости. Координаты $y_i$
находят градиентным спуском.

**Выбор распределения Стьюдента.** В пространстве высокой размерности
у точки много соседей на умеренном расстоянии, а на плоскости столько точек
на таком расстоянии не помещается. Функция $(1 + d^2)^{-1}$ убывает медленнее
гауссовой $\exp(-d^2/2)$, поэтому одному и тому же значению сходства
на плоскости соответствует большее расстояние, и умеренно далекие объекты
располагаются дальше друг от друга. В результате группы близких объектов
разделяются промежутками. Для уровня сходства 0.1: $\exp(-d^2/2) = 0.1$ при
$d = \sqrt{2 \ln 10} \approx 2.15$, $(1 + d^2)^{-1} = 0.1$ при $d = 3$.

In [ ]:
distance = np.linspace(0, 5, 300)
gauss_kernel = np.exp(-distance ** 2 / 2)
student_kernel = 1 / (1 + distance ** 2)
plt.figure(figsize=(8, 4))
plt.plot(distance, gauss_kernel, color=BLUE, lw=2, label="гауссово ядро exp(−d²/2)")
plt.plot(distance, student_kernel, color=OCHRE, lw=2, label="распределение Стьюдента (1 + d²)⁻¹")
plt.axhline(0.1, color=GREY, ls="--", lw=1)
for d_value, color in [(np.sqrt(2 * np.log(10)), BLUE), (3.0, OCHRE)]:
    plt.scatter(d_value, 0.1, color=color, zorder=3)
    plt.annotate(f"d = {d_value:.2f}", (d_value, 0.1), xytext=(5, 8), textcoords="offset points", color=color)
plt.xlabel("расстояние d")
plt.ylabel("сходство")
plt.title("Значению сходства 0.1 соответствуют разные расстояния")
plt.legend()
plt.show()

Метод t-SNE на 2500 изображениях при `perplexity` = 30.

In [ ]:
start = time.perf_counter()
tsne_points = TSNE(n_components=2, perplexity=30, random_state=SEED).fit_transform(vis_pca50)
tsne_seconds = time.perf_counter() - start
fig, ax = plt.subplots(figsize=(11, 9))
plot_digit_map(ax, tsne_points, vis_y, vis_X, f"t-SNE, perplexity = 30, время {tsne_seconds:.1f} с")
plt.show()

In [ ]:
interactive_digit_map(tsne_points, vis_y, "t-SNE, perplexity = 30")

Изображения одной цифры образуют отдельные группы. Группы цифр 4 и 9
перемешаны, рядом с ними расположена группа цифры 7; соприкасаются также
группы цифр 3, 5 и 8. Эти цифры сходны по начертанию.

**Влияние перплексии.** При малом значении `perplexity` распределение $P_i$
сосредоточено на нескольких ближайших объектах, и метод сохраняет только
самые локальные связи: группы распадаются на мелкие части. При большом
значении учитывается больше соседей, и сохраняется более крупная структура.

In [ ]:
def plot_labeled_map(ax, points, labels, title):
    for digit_class in range(10):
        mask = labels == digit_class
        ax.scatter(points[mask, 0], points[mask, 1], s=3, color=CLUSTER_COLORS[digit_class], alpha=0.6)
        center = np.median(points[mask], axis=0)
        ax.text(center[0], center[1], str(digit_class), fontsize=13, weight="bold", ha="center", va="center",
                bbox={"facecolor": "white", "alpha": 0.7, "edgecolor": "none", "pad": 1})
    ax.set_title(title, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])


fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, perplexity in zip(axes, [5, 30, 100]):
    points = TSNE(n_components=2, perplexity=perplexity, random_state=SEED).fit_transform(vis_pca50)
    plot_labeled_map(ax, points, vis_y, f"t-SNE, perplexity = {perplexity}")
plt.tight_layout()
plt.show()

### Равномерная аппроксимация и проекция многообразия (UMAP)

Метод UMAP также сохраняет соседей, но строит не распределения вероятностей,
а взвешенный граф ближайших соседей, и оптимизирует другую функцию потерь.

**Граф в исходном пространстве.** Для каждого объекта $x_i$ находят $k$ ближайших
соседей, $k$ = `n_neighbors`. Пусть $\rho_i$ — расстояние до ближайшего соседа.
Вес ребра от $x_i$ к соседу $x_j$:

$$w_{j \mid i} = \exp\Big(-\frac{\rho(x_i, x_j) - \rho_i}{\sigma_i}\Big),$$

где $\sigma_i$ находят из условия

$$\sum_{j \in \mathrm{kNN}(i)} w_{j \mid i} = \log_2 k.$$

Вычитание $\rho_i$ гарантирует, что у каждого объекта хотя бы одно ребро
имеет вес 1. Условие на сумму играет ту же роль, что перплексия в t-SNE:
окрестность каждой точки нормируется по ее локальной плотности, и параметр
`n_neighbors` определяет размер окрестности так же, как `perplexity`.
Веса симметризуют как вероятность того, что ребро есть хотя бы в одном
направлении:

$$w_{ij} = w_{j \mid i} + w_{i \mid j} - w_{j \mid i}\, w_{i \mid j}.$$

На схеме граф для 40 точек на спирали при $k = 5$; толщина ребра пропорциональна $w_{ij}$.

In [ ]:
def fuzzy_graph(X, k):
    """Симметричные веса ребер графа k ближайших соседей, как в UMAP."""
    distances, neighbors = NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X)
    distances, neighbors = distances[:, 1:], neighbors[:, 1:]
    directed = np.zeros((len(X), len(X)))
    for i in range(len(X)):
        rho = distances[i, 0]
        low, high = 1e-4, 1e4
        for _ in range(80):
            sigma = np.sqrt(low * high)
            total = np.exp(-(distances[i] - rho) / sigma).sum()
            if total > np.log2(k):
                high = sigma
            else:
                low = sigma
        directed[i, neighbors[i]] = np.exp(-(distances[i] - rho) / sigma)
    weights = directed + directed.T - directed * directed.T
    return weights


t_curve = np.sort(np.random.default_rng(2).uniform(0, 3 * np.pi, 40))
curve_points = np.c_[t_curve * np.cos(t_curve), t_curve * np.sin(t_curve)] / 3 + np.random.default_rng(3).normal(0, 0.08, (40, 2))
graph_weights = fuzzy_graph(curve_points, k=5)
fig, ax = plt.subplots(figsize=(7, 6))
for i, j in zip(*np.nonzero(np.triu(graph_weights))):
    ax.plot(*curve_points[[i, j]].T, color=BLUE, lw=4 * graph_weights[i, j], alpha=0.7)
ax.scatter(curve_points[:, 0], curve_points[:, 1], s=40, color=OCHRE, zorder=3)
ax.set_aspect("equal")
ax.set_title("Взвешенный граф пяти ближайших соседей")
ax.set_xticks([])
ax.set_yticks([])
plt.show()

**Сходство на плоскости** для образов $y_i, y_j$:

$$v_{ij} = \big(1 + a\,\|y_i - y_j\|^{2b}\big)^{-1}.$$

Параметры $a$ и $b$ вычисляют по параметру `min_dist` — наименьшему расстоянию
между образами, при котором сходство еще близко к 1. При $a = b = 1$ формула
совпадает с распределением Стьюдента из t-SNE.

**Функция потерь** — перекрестная энтропия (cross-entropy) между весами ребер:

$$\sum_{i \neq j} \Big[\, w_{ij} \ln \frac{w_{ij}}{v_{ij}} + (1 - w_{ij}) \ln \frac{1 - w_{ij}}{1 - v_{ij}} \Big].$$

Первое слагаемое велико, если соседи в графе ($w_{ij}$ близко к 1) далеки на плоскости:
оно притягивает соседей. Второе велико, если не соседние в графе объекты
($w_{ij}$ близко к 0) оказались рядом: оно отталкивает их. Явное отталкивающее
слагаемое отличает UMAP от t-SNE, где разнесение групп обеспечивается только
нормировкой $q_{ij}$. На рисунке функция $v(d)$ при двух значениях `min_dist`.

In [ ]:
plt.figure(figsize=(8, 4))
for min_dist, color in [(0.1, BLUE), (0.5, OCHRE)]:
    a, b = umap_internal.find_ab_params(1.0, min_dist)
    plt.plot(distance, 1 / (1 + a * distance ** (2 * b)), color=color, lw=2, label=f"min_dist = {min_dist}: a = {a:.2f}, b = {b:.2f}")
plt.plot(distance, student_kernel, color=GREY, ls="--", label="a = b = 1, как в t-SNE")
plt.xlabel("расстояние между образами")
plt.ylabel("сходство v")
plt.legend()
plt.show()

Метод UMAP на тех же 2500 изображениях при `n_neighbors` = 15. Первый вызов
включает компиляцию кода библиотекой numba, поэтому занимает больше времени.

In [ ]:
start = time.perf_counter()
umap_points = umap.UMAP(n_neighbors=15, random_state=SEED).fit_transform(vis_pca50)
umap_seconds = time.perf_counter() - start
fig, ax = plt.subplots(figsize=(11, 9))
plot_digit_map(ax, umap_points, vis_y, vis_X, f"UMAP, n_neighbors = 15, время {umap_seconds:.1f} с")
plt.show()

In [ ]:
interactive_digit_map(umap_points, vis_y, "UMAP, n_neighbors = 15")

**Влияние числа соседей.** При малом `n_neighbors` граф описывает только ближайшую
окрестность: группы компактны и разнесены, их взаимное расположение
случайно. При большом значении граф связывает больше объектов, сохраняется
взаимное расположение групп, и сходные цифры сближаются: при `n_neighbors` = 100
группы цифр 3, 5 и 8, а также 4, 7 и 9 образуют связные области.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, n_neighbors in zip(axes, [5, 15, 100]):
    points = umap.UMAP(n_neighbors=n_neighbors, random_state=SEED).fit_transform(vis_pca50)
    plot_labeled_map(ax, points, vis_y, f"UMAP, n_neighbors = {n_neighbors}")
plt.tight_layout()
plt.show()

### Сравнение трех методов

Качество двумерного представления оценивают **надежностью** (trustworthiness).
Пусть $r(i, j)$ — номер объекта $j$ в списке соседей объекта $i$ в исходном
пространстве, упорядоченном по возрастанию расстояния, а $U_k(i)$ — множество
объектов, которые входят в $k$ ближайших соседей $i$ на плоскости, но не входят
в $k$ ближайших соседей в исходном пространстве. Тогда

$$T(k) = 1 - \frac{2}{n k (2n - 3k - 1)} \sum_{i=1}^{n} \sum_{j \in U_k(i)} \big(r(i, j) - k\big).$$

Каждый ложный сосед штрафуется тем сильнее, чем дальше он в действительности;
$T(k) = 1$, если все соседи на плоскости являются соседями и в исходном
пространстве. Вторая величина — accuracy метода k ближайших соседей,
обученного на двумерных координатах, по кросс-валидации: она показывает,
насколько разделены цифры на рисунке.

Три представления на одних 2500 изображениях.

In [ ]:
vis_embeddings = {
    "PCA, две компоненты": PCA(n_components=2, random_state=SEED).fit_transform(vis_pca50),
    "t-SNE, perplexity = 30": tsne_points,
    "UMAP, n_neighbors = 15": umap_points,
}
fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
for ax, (name, points) in zip(axes, vis_embeddings.items()):
    plot_digit_map(ax, points, vis_y, vis_X, name, n_marks=25)
    ax.get_legend().remove()
handles = [Line2D([], [], marker="o", ls="", color=CLUSTER_COLORS[d], label=str(d)) for d in range(10)]
fig.legend(handles=handles, title="цифра", loc="center right", fontsize=9)
plt.tight_layout(rect=(0, 0, 0.96, 1))
plt.show()

В таблице строки — методы, столбцы — надежность $T(10)$ и accuracy метода
k ближайших соседей на двумерных координатах.

In [ ]:
vis_rows = {}
for name, points in vis_embeddings.items():
    vis_rows[name] = {
        "надежность T(10)": trustworthiness(vis_pca50, points, n_neighbors=10),
        "accuracy kNN на плоскости": cross_val_score(KNeighborsClassifier(n_neighbors=5), points, vis_y, cv=5).mean(),
    }
show_table(pd.DataFrame(vis_rows).T)

t-SNE и UMAP сохраняют соседей почти без ошибок, а две главные компоненты
сохраняют их плохо: PCA максимизирует дисперсию, а не сохранение окрестностей.

Двумерные представления t-SNE и UMAP интерпретируют со следующими ограничениями:

- соседство точек на рисунке отражает соседство в исходном пространстве,
  так как именно его оптимизирует функция потерь;
- размеры групп и расстояния между ними на рисунке t-SNE не соответствуют
  исходным: подбор $\sigma_i$ выравнивает плотность, а функция потерь
  слабо зависит от далеких пар; у UMAP взаимное расположение групп
  сохраняется лучше, но также неточно;
- результат зависит от `random_state` и параметров, поэтому вывод, который
  не воспроизводится при другом начальном состоянии, не используют;
- у t-SNE нет преобразования новых объектов, поэтому его координаты
  не применяют как признаки модели; выводы, полученные по рисунку,
  проверяют на исходных данных.

## 5. Кластеризация

Задача кластеризации (clustering) — разбить объекты на группы так, чтобы объекты внутри
группы были сходны, а объекты разных групп различались; правильные ответы
неизвестны. Для изучения методов используют пять двумерных наборов
из sklearn, в которых истинное разбиение известно: сферические кластеры,
вытянутые кластеры, кластеры разной плотности, полумесяцы и кольца.
Все наборы стандартизованы.

In [ ]:
X_aniso, y_aniso = make_blobs(n_samples=500, centers=3, random_state=170)
X_aniso = X_aniso @ np.array([[0.6, -0.64], [-0.41, 0.85]])
X_density, y_density = make_blobs(n_samples=500, centers=3, cluster_std=[1.0, 2.5, 0.5], random_state=170)

datasets = {
    "сферические": make_blobs(n_samples=500, centers=3, random_state=8),
    "вытянутые": (X_aniso, y_aniso),
    "разная плотность": (X_density, y_density),
    "полумесяцы": make_moons(n_samples=500, noise=0.06, random_state=SEED),
    "кольца": make_circles(n_samples=500, factor=0.5, noise=0.05, random_state=SEED),
}
datasets = {name: (StandardScaler().fit_transform(X), y) for name, (X, y) in datasets.items()}


def plot_partition(ax, X, labels, title):
    labels = np.asarray(labels)
    colors = np.where(labels >= 0, CLUSTER_COLORS[labels % len(CLUSTER_COLORS)], "#111111")
    ax.scatter(X[:, 0], X[:, 1], s=7, c=colors)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])


fig, axes = plt.subplots(1, 5, figsize=(17, 3.4))
for ax, (name, (X, y)) in zip(axes, datasets.items()):
    plot_partition(ax, X, y, f"{name}: истинное разбиение")
plt.tight_layout()
plt.show()

### Метрики качества при известном разбиении

Номера кластеров произвольны: разбиения $\{0, 0, 1\}$ и $\{1, 1, 0\}$ совпадают. Поэтому
метрики сравнивают не номера, а то, какие объекты отнесены к одному кластеру.

**Индекс Рэнда** (Rand index, RI). Рассматривают все $\binom{n}{2}$ пар объектов.
Пусть $a$ — число пар, отнесенных к одному кластеру в обоих разбиениях,
$b$ — число пар, отнесенных к разным кластерам в обоих разбиениях. Индекс
Рэнда — доля согласованных пар:

$$\mathrm{RI} = \frac{a + b}{\binom{n}{2}}.$$

Пример: шесть объектов, истинное разбиение $\{0, 0, 0, 1, 1, 1\}$, найденное
$\{0, 0, 1, 1, 2, 2\}$. На схеме объекты расположены по истинным классам
(рамки), цвет — найденный кластер; линии соединяют пары, отнесенные
к одному кластеру в найденном разбиении. В таблице все 15 пар и их статус.

In [ ]:
truth_small = np.array([0, 0, 0, 1, 1, 1])
found_small = np.array([0, 0, 1, 1, 2, 2])
toy_xy = np.array([[0, 0], [1, 0], [2, 0], [4, 0], [5, 0], [6, 0]], dtype=float)
fig, ax = plt.subplots(figsize=(8, 2.2))
for left, right in [(-0.5, 2.5), (3.5, 6.5)]:
    ax.add_patch(Rectangle((left, -0.6), right - left, 1.2, fill=False, lw=1.2, color=GREY))
for i, j in itertools.combinations(range(6), 2):
    if found_small[i] == found_small[j]:
        ax.plot(*toy_xy[[i, j]].T, color=BLACK, lw=2, zorder=1)
ax.scatter(toy_xy[:, 0], toy_xy[:, 1], s=500, c=CLUSTER_COLORS[found_small], zorder=2)
for i, (x, y) in enumerate(toy_xy):
    ax.text(x, y, str(i), ha="center", va="center", color="white", fontsize=12, weight="bold", zorder=3)
ax.text(1, 0.8, "истинный класс 0", ha="center", color=GREY)
ax.text(5, 0.8, "истинный класс 1", ha="center", color=GREY)
ax.set_xlim(-1, 7)
ax.set_ylim(-1, 1.2)
ax.axis("off")
plt.show()

In [ ]:
pair_rows = []
for i, j in itertools.combinations(range(6), 2):
    together_truth, together_found = truth_small[i] == truth_small[j], found_small[i] == found_small[j]
    if together_truth and together_found:
        status = "a: вместе в обоих"
    elif not together_truth and not together_found:
        status = "b: врозь в обоих"
    else:
        status = "не согласована"
    pair_rows.append({"пара": f"({i}, {j})", "истинное: вместе": together_truth, "найденное: вместе": together_found, "статус": status})
pairs_table = pd.DataFrame(pair_rows)
counts = pairs_table["статус"].value_counts()
rand_index = (counts.get("a: вместе в обоих", 0) + counts.get("b: врозь в обоих", 0)) / comb(6, 2)
print(counts.to_string())
print(f"RI = (a + b) / C(6, 2) = {rand_index:.3f}")
pairs_table

**Скорректированный индекс Рэнда** (adjusted Rand index, ARI). У случайного
разбиения индекс Рэнда заметно больше нуля, так как большинство пар в любом
разбиении отнесены к разным кластерам. Поправку строят по таблице
сопряженности (contingency table): $n_{ij}$ — число объектов в истинном классе $i$ и найденном
кластере $j$, $a_i = \sum_j n_{ij}$, $b_j = \sum_i n_{ij}$. Число пар, отнесенных к одному
кластеру в обоих разбиениях, равно $\sum_{ij}\binom{n_{ij}}{2}$. Его математическое
ожидание при случайной перестановке меток с сохранением размеров кластеров:

$$E = \frac{\sum_i \binom{a_i}{2} \sum_j \binom{b_j}{2}}{\binom{n}{2}}.$$

Скорректированный индекс вычитает ожидание и нормирует на наибольшее
возможное значение:

$$\mathrm{ARI} = \frac{\sum_{ij}\binom{n_{ij}}{2} - E}{\frac{1}{2}\Big[\sum_i \binom{a_i}{2} + \sum_j \binom{b_j}{2}\Big] - E}.$$

$\mathrm{ARI} = 1$ при совпадении разбиений, около 0 — для случайного разбиения;
отрицательные значения означают согласованность ниже случайной. Вычисление
для примера по формуле и в sklearn.

In [ ]:
contingency_small = pd.crosstab(truth_small, found_small)
sum_cells = sum(comb(int(v), 2) for v in contingency_small.to_numpy().ravel())
sum_rows = sum(comb(int(v), 2) for v in contingency_small.sum(axis=1))
sum_cols = sum(comb(int(v), 2) for v in contingency_small.sum(axis=0))
expected = sum_rows * sum_cols / comb(6, 2)
ari_by_hand = (sum_cells - expected) / (0.5 * (sum_rows + sum_cols) - expected)
print(contingency_small)
print(f"Σ C(n_ij, 2) = {sum_cells}, Σ C(a_i, 2) = {sum_rows}, Σ C(b_j, 2) = {sum_cols}, E = {expected:.3f}")
print(f"ARI по формуле: {ari_by_hand:.4f}, adjusted_rand_score: {adjusted_rand_score(truth_small, found_small):.4f}")

**Энтропийные метрики** (entropy-based metrics). Пусть $C$ — истинные классы, $K$ — найденные кластеры,
$H(C) = -\sum_c \frac{n_c}{n} \ln \frac{n_c}{n}$ — энтропия разбиения на классы,
$H(C \mid K)$ — условная энтропия классов при известном кластере. Определяют:

- **однородность** (homogeneity) $h = 1 - \frac{H(C \mid K)}{H(C)}$; $h = 1$, если каждый кластер содержит
  объекты только одного класса, так как тогда $H(C \mid K) = 0$;
- **полноту** (completeness) $c = 1 - \frac{H(K \mid C)}{H(K)}$; $c = 1$, если каждый класс целиком
  попал в один кластер;
- **V-меру** (V-measure) $\frac{2hc}{h + c}$ — среднее гармоническое однородности и полноты;
  нормированная взаимная информация (normalized mutual information, NMI)
  в sklearn по умолчанию совпадает с ней.

### Метрики качества при неизвестном разбиении

- **Силуэт** (silhouette). Для объекта $i$: $a(i)$ — среднее расстояние до объектов его кластера,
  $b(i)$ — наименьшее среди чужих кластеров среднее расстояние до их объектов,
  $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))} \in [-1, 1]$. Метрика — среднее $s(i)$ по выборке.
  Значения, близкие к 1, означают, что объект близок к своему кластеру и далек
  от остальных.
- **Индекс Калински — Харабаша** (Calinski — Harabasz index) $\frac{\mathrm{tr}(B) / (k - 1)}{\mathrm{tr}(W) / (n - k)}$, где $B$ и $W$ —
  матрицы межкластерного и внутрикластерного рассеяния; большее значение лучше.
- **Индекс Дэвиса — Болдина** (Davies — Bouldin index) $\frac{1}{k}\sum_i \max_{j \neq i} \frac{s_i + s_j}{\rho(\mu_i, \mu_j)}$, где $s_i$ —
  среднее расстояние объектов кластера $i$ до его центра $\mu_i$; меньшее значение лучше.

### Метрики на примерах

На наборе сферических кластеров сравнивают пять разбиений: истинное,
истинное с переставленными номерами, разбиение со слитыми кластерами,
разбиение с разрезанными кластерами и случайное. В заголовках рисунков —
ARI и силуэт.

In [ ]:
def clustering_scores(X, y_true, labels):
    labels = np.asarray(labels)
    n_clusters = len(set(labels) - {-1})
    defined = 1 < n_clusters < len(X)
    scores = {
        "ARI": adjusted_rand_score(y_true, labels),
        "NMI": normalized_mutual_info_score(y_true, labels),
        "однородность": homogeneity_score(y_true, labels),
        "полнота": completeness_score(y_true, labels),
        "V-мера": v_measure_score(y_true, labels),
        "силуэт": silhouette_score(X, labels) if defined else np.nan,
        "Калински — Харабаш": calinski_harabasz_score(X, labels) if defined else np.nan,
        "Дэвис — Болдин": davies_bouldin_score(X, labels) if defined else np.nan,
    }
    return scores


X_balls, y_balls = datasets["сферические"]
split_halves = y_balls * 2 + (X_balls[:, 0] > np.median(X_balls[:, 0])).astype(int)
variants = {
    "истинное": y_balls,
    "номера переставлены": (y_balls + 1) % 3,
    "два кластера слиты": np.where(y_balls == 2, 1, y_balls),
    "кластеры разрезаны": split_halves,
    "случайное": np.random.default_rng(0).integers(0, 3, len(y_balls)),
}
metric_demo = pd.DataFrame({name: clustering_scores(X_balls, y_balls, labels) for name, labels in variants.items()}).T

fig, axes = plt.subplots(1, 5, figsize=(18, 3.6))
for ax, (name, labels) in zip(axes, variants.items()):
    plot_partition(ax, X_balls, labels, f"{name}\nARI {metric_demo.loc[name, 'ARI']:.2f}, силуэт {metric_demo.loc[name, 'силуэт']:.2f}")
plt.tight_layout()
plt.show()

В таблице строки — разбиения, столбцы — метрики; заливка по столбцам,
для индекса Дэвиса — Болдина ярче меньшее значение.

In [ ]:
show_table(metric_demo)

Свойства метрик на примерах:

- перестановка номеров не меняет ни одну метрику, так как метрики зависят
  только от того, какие объекты отнесены к одному кластеру;
- при слиянии двух кластеров полнота равна 1 (каждый класс целиком в одном
  кластере), а однородность снижается (слитый кластер содержит два класса);
- при разрезании кластеров, наоборот, однородность равна 1, а полнота снижается;
- у случайного разбиения ARI близок к нулю, силуэт отрицателен;
- метрики без истинного разбиения оценивают форму кластеров и максимальны
  для компактных выпуклых кластеров, поэтому на полумесяцах истинное
  разбиение имеет низкий силуэт.

### Метод k-средних

Метод k-средних (k-means, класс `KMeans`) ищет $k$ центров $\mu_1, \dots, \mu_k$ и относит каждый объект
к ближайшему центру так, чтобы минимизировать **инерцию** (inertia) — сумму квадратов
расстояний объектов до центров их кластеров:

$$J = \sum_{i=1}^{n} \|x_i - \mu_{c(i)}\|^2.$$

**Алгоритм Ллойда** (Lloyd's algorithm) чередует два шага:

1. отнесение: каждый объект относят к ближайшему центру; каждое слагаемое $J$
   при этом не возрастает;
2. пересчет: каждый центр заменяют средним его объектов; сумма
   $\sum_{i \in K} \|x_i - \mu\|^2$ как функция $\mu$ имеет градиент $-2\sum_{i \in K}(x_i - \mu)$,
   который обращается в ноль при $\mu = \frac{1}{|K|}\sum_{i \in K} x_i$, поэтому
   и этот шаг не увеличивает $J$.

Величина $J$ не возрастает и ограничена снизу нулем, поэтому алгоритм сходится,
но к локальному минимуму, который зависит от начальных центров.

На рисунке четыре итерации на наборе сферических кластеров; два из трех
начальных центров выбраны в одном кластере. Столбцы — номера итераций.
В верхнем ряду — шаг отнесения: фон окрашен по ближайшему центру,
крестами отмечены текущие центры. В нижнем ряду — шаг пересчета:
серые кресты — центры до пересчета, черные — после, стрелки показывают
их перемещение. В заголовках — значение $J$ после шага.

In [ ]:
start_indices = np.r_[np.flatnonzero(y_balls == 0)[:2], np.flatnonzero(y_balls == 1)[:1]]
centers = X_balls[start_indices].copy()
grid_x, grid_y = np.meshgrid(np.linspace(X_balls[:, 0].min() - 0.5, X_balls[:, 0].max() + 0.5, 300),
                             np.linspace(X_balls[:, 1].min() - 0.5, X_balls[:, 1].max() + 0.5, 300))
grid = np.c_[grid_x.ravel(), grid_y.ravel()]
fig, axes = plt.subplots(2, 4, figsize=(18, 8.5))
for step in range(4):
    distances = ((X_balls[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    assignment = distances.argmin(axis=1)
    inertia_assigned = distances[np.arange(len(X_balls)), assignment].sum()
    regions = ((grid[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2).argmin(axis=1).reshape(grid_x.shape)
    top = axes[0, step]
    top.contourf(grid_x, grid_y, regions, levels=[-0.5, 0.5, 1.5, 2.5], colors=list(CLUSTER_COLORS[:3]), alpha=0.15)
    plot_partition(top, X_balls, assignment, f"итерация {step + 1}, отнесение: J = {inertia_assigned:.1f}")
    top.scatter(centers[:, 0], centers[:, 1], s=220, marker="X", color=BLACK)

    new_centers = np.array([X_balls[assignment == j].mean(axis=0) for j in range(3)])
    inertia_updated = ((X_balls - new_centers[assignment]) ** 2).sum()
    bottom = axes[1, step]
    plot_partition(bottom, X_balls, assignment, f"итерация {step + 1}, пересчет: J = {inertia_updated:.1f}")
    bottom.scatter(centers[:, 0], centers[:, 1], s=120, marker="X", color=GREY)
    bottom.scatter(new_centers[:, 0], new_centers[:, 1], s=220, marker="X", color=BLACK)
    for old, new in zip(centers, new_centers):
        bottom.annotate("", xy=new, xytext=old, arrowprops={"arrowstyle": "->", "color": BLACK, "lw": 1.5})
    centers = new_centers
axes[0, 0].set_ylabel("отнесение к центрам", fontsize=12)
axes[1, 0].set_ylabel("пересчет центров", fontsize=12)
plt.tight_layout()
plt.show()

**Выбор начальных центров.** При случайном выборе два начальных центра могут
оказаться в одном кластере, и алгоритм сходится к плохому локальному минимуму.
Метод **k-means++** выбирает первый центр случайно, а каждый следующий —
с вероятностью, пропорциональной квадрату расстояния до ближайшего из уже
выбранных центров; в результате начальные центры распределены по данным.
На наборе из десяти кластеров сравнивают инерцию после одного запуска
со случайными центрами и с k-means++, по 100 запусков каждого способа.

In [ ]:
X_ten, _ = make_blobs(n_samples=1500, centers=10, cluster_std=0.6, random_state=3)
init_inertia = {
    init: [KMeans(n_clusters=10, init=init, n_init=1, random_state=seed).fit(X_ten).inertia_ for seed in range(100)]
    for init in ["random", "k-means++"]
}
plt.figure(figsize=(8, 3.8))
for (init, values), color in zip(init_inertia.items(), [GREY, BLUE]):
    plt.hist(values, bins=30, alpha=0.7, color=color, label=f"{init}: медиана {np.median(values):.0f}")
plt.xlabel("инерция после одного запуска")
plt.ylabel("число запусков")
plt.legend()
plt.show()

При k-means++ инерция в среднем меньше: алгоритм реже останавливается в плохом
минимуме. В `KMeans` по умолчанию используют k-means++, а параметр `n_init`
задает число запусков с разными начальными центрами, из которых оставляют
результат с наименьшей инерцией.

**Выбор числа кластеров.** Инерция убывает с ростом $k$ всегда и равна нулю при $k = n$.
Поэтому строят график инерции от $k$ и выбирают значение, после которого
убывание резко замедляется (метод локтя, elbow method), либо значение с наибольшим силуэтом.
Пример — набор из четырех кластеров.

In [ ]:
X_four, _ = make_blobs(n_samples=600, centers=4, cluster_std=1.0, random_state=11)
k_values = range(2, 10)
k_fits = [KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_four) for k in k_values]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(k_values, [fit.inertia_ for fit in k_fits], "o-", color=BLUE)
axes[0].set_ylabel("инерция")
axes[1].plot(k_values, [silhouette_score(X_four, fit.labels_) for fit in k_fits], "o-", color=OCHRE)
axes[1].set_ylabel("силуэт")
for ax in axes:
    ax.axvline(4, color=GREY, ls="--", lw=1)
    ax.set_xlabel("число кластеров k")
plt.tight_layout()
plt.show()

Излом инерции и максимум силуэта приходятся на $k = 4$. На реальных данных
эти критерии часто не совпадают, и число кластеров выбирают также
из содержательных соображений.

**Ограничения метода.** Инерция — сумма квадратов расстояний до центров,
поэтому метод k-средних находит выпуклые кластеры близкого размера. На рисунке
результат метода с истинным числом кластеров на всех пяти наборах.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(17, 3.4))
for ax, (name, (X, y)) in zip(axes, datasets.items()):
    labels = KMeans(n_clusters=len(np.unique(y)), n_init=10, random_state=SEED).fit_predict(X)
    plot_partition(ax, X, labels, f"{name}: ARI {adjusted_rand_score(y, labels):.2f}")
plt.tight_layout()
plt.show()

Сферические кластеры найдены верно. Граница метода k-средних между двумя
кластерами — серединный перпендикуляр к отрезку между их центрами,
поэтому вытянутые кластеры, полумесяцы и кольца разделены прямыми линиями
неверно. У кластеров разной плотности часть точек широкого кластера
отнесена к соседним, так как граница проходит посередине между центрами
независимо от разброса кластеров.

### Агломеративная кластеризация (agglomerative clustering)

Алгоритм начинает с разбиения, в котором каждый объект образует отдельный
кластер. На каждом шаге объединяют два кластера с наименьшим расстоянием
между ними, пока не останется $k$ кластеров. Расстояние между кластерами $A$ и $B$
определяют одним из способов:

- метод одиночной связи (single linkage): $d(A, B) = \min_{a \in A,\, b \in B} \rho(a, b)$;
- метод полной связи (complete linkage): $d(A, B) = \max_{a \in A,\, b \in B} \rho(a, b)$;
- метод средней связи (average linkage): $d(A, B) = \frac{1}{|A|\,|B|} \sum_{a \in A} \sum_{b \in B} \rho(a, b)$;
- метод Уорда (Ward's method): $d(A, B) = \frac{|A|\,|B|}{|A| + |B|}\,\|\mu_A - \mu_B\|^2$ — прирост суммы квадратов
  расстояний до центров при объединении $A$ и $B$.

Метод Уорда на каждом шаге выбирает объединение с наименьшим приростом
инерции, то есть минимизирует тот же критерий, что и метод k-средних,
поэтому дает сходные разбиения. Последовательность объединений изображают
**дендрограммой** (dendrogram): по горизонтали — объекты, высота горизонтального отрезка
равна расстоянию, при котором объединились два кластера; разрез дендрограммы
на заданной высоте дает разбиение. На рисунке дендрограмма метода Уорда
для 40 объектов набора сферических кластеров.

In [ ]:
sample = X_balls[np.random.default_rng(1).choice(len(X_balls), 40, replace=False)]
merges = linkage(sample, method="ward")
plt.figure(figsize=(11, 4))
dendrogram(merges, color_threshold=6, no_labels=True)
plt.axhline(6, color=OCHRE, ls="--", label="разрез: три кластера")
plt.ylabel("расстояние объединения, метод Уорда")
plt.legend()
plt.show()

Четыре способа определения расстояния сравнивают на двух наборах. Строки
рисунка соответствуют наборам: верхняя — полумесяцы, нижняя — кластеры
разной плотности. Столбцы соответствуют способам определения расстояния,
слева направо: одиночная, полная, средняя связь и метод Уорда. Цвет точки —
найденный кластер, в заголовке — ARI.

In [ ]:
linkage_names = {"single": "одиночная связь (single)", "complete": "полная связь (complete)",
                 "average": "средняя связь (average)", "ward": "метод Уорда (Ward)"}
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, name in enumerate(["полумесяцы", "разная плотность"]):
    X, y = datasets[name]
    for col, (method, method_name) in enumerate(linkage_names.items()):
        labels = AgglomerativeClustering(n_clusters=len(np.unique(y)), linkage=method).fit_predict(X)
        plot_partition(axes[row, col], X, labels, f"{method_name}: ARI {adjusted_rand_score(y, labels):.2f}")
    axes[row, 0].set_ylabel(name, fontsize=12)
plt.tight_layout()
plt.show()

Метод одиночной связи определяет расстояние между кластерами по ближайшей
паре точек, поэтому верно разделяет полумесяцы: соседние точки каждой дуги
близки, а наименьшее расстояние между дугами больше. На кластерах разной
плотности он дает ARI около нуля: в широком кластере есть отдельные точки,
удаленные от остальных, и они остаются отдельными кластерами до последних
шагов, а основные кластеры объединяются раньше. Методы полной, средней связи
и Уорда учитывают все точки кластеров, предпочитают компактные кластеры
и разделяют полумесяцы неверно.

### Плотностная кластеризация с выделением шума (DBSCAN)

Метод DBSCAN выделяет кластеры как области высокой плотности. Для точки $x$
определяют ее $\varepsilon$-окрестность ($\varepsilon$-neighborhood):

$$N_\varepsilon(x) = \{x_j : \rho(x, x_j) \le \varepsilon\}.$$

Точки делят на три типа:

- ядровая точка (core point): $|N_\varepsilon(x)| \ge$ `min_samples`, точка $x$ при подсчете учитывается;
- граничная точка (border point): не является ядровой, но лежит в $\varepsilon$-окрестности ядровой точки;
- шумовая точка (noise point): остальные точки, им присваивают метку $-1$.

Кластер — максимальное множество ядровых точек, в котором любые две точки
соединены цепочкой ядровых точек с соседними элементами на расстоянии
не больше $\varepsilon$, вместе с граничными точками этих ядровых точек. Число
кластеров заранее не задают, его определяет алгоритм. Шумовые точки ни к одному
кластеру не относятся.

**Выбор $\varepsilon$.** Для каждой точки $x$ вычисляют расстояние $d_k(x)$ до ее $k$-го
ближайшего соседа при $k$ = `min_samples` и строят график значений $d_k$,
упорядоченных по возрастанию. Внутри кластеров $d_k$ мало, у шумовых точек оно
резко возрастает. Значение $\varepsilon$ выбирают в области излома графика.

In [ ]:
X_moon_clusters, y_moon_clusters = datasets["полумесяцы"]
min_samples = 5
neighbor_distances, _ = NearestNeighbors(n_neighbors=min_samples).fit(X_moon_clusters).kneighbors(X_moon_clusters)
kth_distance = np.sort(neighbor_distances[:, -1])
chosen_eps = 0.2

plt.figure(figsize=(8, 3.6))
plt.plot(kth_distance, color=BLUE, lw=2)
plt.axhline(chosen_eps, color=OCHRE, ls="--", label=f"ε = {chosen_eps}")
plt.xlabel("номер точки после упорядочения")
plt.ylabel(f"d_k, k = {min_samples}")
plt.legend()
plt.show()

Результат DBSCAN на полумесяцах при трех значениях $\varepsilon$ и `min_samples` = 5;
шумовые точки показаны черным.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, eps in zip(axes, [0.08, chosen_eps, 0.6]):
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_moon_clusters)
    plot_partition(ax, X_moon_clusters, labels,
                   f"ε = {eps}: кластеров {len(set(labels) - {-1})}, шум {np.mean(labels == -1):.0%}, "
                   f"ARI {adjusted_rand_score(y_moon_clusters, labels):.2f}")
plt.tight_layout()
plt.show()

При $\varepsilon$ = 0.08 окрестности малы и лишь часть точек является ядровыми: дуги
распадаются на множество кластеров, значительная доля точек отнесена к шуму.
При $\varepsilon$ = 0.6 окрестности перекрываются между дугами, и все точки образуют
один кластер. Значение $\varepsilon$ = 0.2 из области излома разделяет дуги верно.

### Смесь гауссовых распределений (Gaussian mixture model, GMM)

Модель предполагает, что данные порождены смесью $k$ нормальных распределений:

$$p(x) = \sum_{j=1}^{k} \pi_j \, \mathcal{N}(x \mid \mu_j, \Sigma_j),$$

где $\pi_j$ — доля компоненты, $\mu_j$ — центр, $\Sigma_j$ — ковариационная матрица.
Линии уровня плотности компоненты — эллипсы, форма и ориентация которых
задаются $\Sigma_j$. Параметры находят EM-алгоритмом (expectation — maximization,
ожидание — максимизация), чередующим два шага:

1. E-шаг: для каждого объекта по формуле Байеса вычисляют вероятность
   принадлежности компоненте $j$:
   $r_{ij} = \frac{\pi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}{\sum_{l} \pi_l \mathcal{N}(x_i \mid \mu_l, \Sigma_l)}$;
2. M-шаг: $\pi_j$, $\mu_j$ и $\Sigma_j$ вычисляют как долю, среднее и ковариационную матрицу
   объектов с весами $r_{ij}$.

Отнесение к кластерам мягкое: каждому объекту соответствует вектор вероятностей.
Метод k-средних — предельный случай модели при одинаковых сферических
ковариационных матрицах и отнесении каждого объекта к одной компоненте.
На рисунке метод k-средних и смесь с полными ковариационными матрицами
на вытянутых кластерах; на правой панели — вероятность наиболее вероятной
компоненты для каждой точки.

In [ ]:
def draw_ellipses(ax, gmm):
    for j in range(gmm.n_components):
        values, vectors = np.linalg.eigh(gmm.covariances_[j])
        angle = np.degrees(np.arctan2(vectors[1, -1], vectors[0, -1]))
        for scale in (1, 2):
            width, height = 2 * scale * np.sqrt(values[::-1])
            ax.add_patch(Ellipse(gmm.means_[j], width, height, angle=angle, fill=False, color=BLACK, lw=1))


X_ellipse, y_ellipse = datasets["вытянутые"]
kmeans_labels = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit_predict(X_ellipse)
gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=SEED).fit(X_ellipse)
gmm_labels = gmm.predict(X_ellipse)
confidence = gmm.predict_proba(X_ellipse).max(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
plot_partition(axes[0], X_ellipse, kmeans_labels, f"метод k-средних (k-means): ARI {adjusted_rand_score(y_ellipse, kmeans_labels):.2f}")
plot_partition(axes[1], X_ellipse, gmm_labels, f"смесь гауссовых распределений (GMM): ARI {adjusted_rand_score(y_ellipse, gmm_labels):.2f}")
draw_ellipses(axes[1], gmm)
points = axes[2].scatter(X_ellipse[:, 0], X_ellipse[:, 1], s=7, c=confidence, cmap="viridis")
plt.colorbar(points, ax=axes[2], label="наибольшая вероятность")
axes[2].set_title("вероятность наиболее вероятной компоненты", fontsize=10)
plt.tight_layout()
plt.show()

Эллипсы на средней панели — линии уровня плотности компонент на расстоянии
одного и двух стандартных отклонений. Наименьшая вероятность наиболее
вероятной компоненты — в областях, где компоненты перекрываются.

**Выбор числа компонент** выполняют по байесовскому информационному
критерию (Bayesian information criterion, BIC):

$$\mathrm{BIC} = q \ln n - 2 \ln \hat{L},$$

где $\hat{L}$ — значение функции правдоподобия обученной модели, $q$ — число ее
параметров. Второе слагаемое убывает при улучшении согласия модели
с данными, первое возрастает с числом параметров. Выбирают число компонент
с наименьшим BIC. Пример — набор из четырех кластеров.

In [ ]:
components = range(1, 9)
bic = [GaussianMixture(n_components=k, random_state=SEED).fit(X_four).bic(X_four) for k in components]
best_components = components[int(np.argmin(bic))]
plt.figure(figsize=(8, 3.6))
plt.plot(components, bic, "o-", color=BLUE)
plt.axvline(best_components, color=OCHRE, ls="--", label=f"минимум при {best_components} компонентах")
plt.xlabel("число компонент")
plt.ylabel("BIC")
plt.legend()
plt.show()

### Сравнение методов в одном сетапе

Пять методов применяют к пяти наборам с одинаковыми параметрами: число
кластеров равно истинному, у DBSCAN $\varepsilon$ = 0.2 и `min_samples` = 5. Для каждой
пары «набор — метод» вычисляют все метрики и время обучения вместе
с предсказанием.

In [ ]:
methods = {
    "метод k-средних (k-means)": lambda k: KMeans(n_clusters=k, n_init=10, random_state=SEED),
    "метод Уорда (Ward)": lambda k: AgglomerativeClustering(n_clusters=k, linkage="ward"),
    "метод одиночной связи (single linkage)": lambda k: AgglomerativeClustering(n_clusters=k, linkage="single"),
    "DBSCAN": lambda k: DBSCAN(eps=0.2, min_samples=5),
    "смесь гауссовых распределений (GMM)": lambda k: GaussianMixture(n_components=k, random_state=SEED),
}

comparison_rows, comparison_labels = [], {}
for data_name, (X, y) in datasets.items():
    for method_name, make_method in methods.items():
        start = time.perf_counter()
        labels = make_method(len(np.unique(y))).fit_predict(X)
        seconds = time.perf_counter() - start
        comparison_labels[data_name, method_name] = labels
        comparison_rows.append({"набор": data_name, "метод": method_name, **clustering_scores(X, y, labels), "время, с": seconds})
comparison = pd.DataFrame(comparison_rows)


def comparison_table(values):
    table = comparison.pivot(index="набор", columns="метод", values=values).loc[list(datasets), list(methods)]
    return table

Строки рисунка — наборы данных, столбцы — методы, в заголовке — ARI;
шумовые точки DBSCAN показаны черным.

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(17, 16))
for row, (data_name, (X, y)) in enumerate(datasets.items()):
    for col, method_name in enumerate(methods):
        labels = comparison_labels[data_name, method_name]
        title = f"ARI {adjusted_rand_score(y, labels):.2f}"
        if row == 0:
            title = f"{method_name}\n{title}"
        plot_partition(axes[row, col], X, labels, title)
    axes[row, 0].set_ylabel(data_name, fontsize=12)
plt.tight_layout()
plt.show()

**Таблица ARI.** Строки — наборы данных, столбцы — методы; значение 1 означает
совпадение с истинным разбиением, значение около 0 — согласованность
на уровне случайного разбиения. Шкала заливки общая для всей таблицы.

In [ ]:
show_table(comparison_table("ARI"), axis=None)

**Таблица силуэтов.** Строки и столбцы те же; силуэт вычислен по найденному
разбиению без использования истинных меток.

In [ ]:
show_table(comparison_table("силуэт"), axis=None)

**Таблица времени** обучения вместе с предсказанием, в миллисекундах. Строки
и столбцы те же; ярче меньшее время. На наборах из 500 точек все методы
работают за миллисекунды; различия существенны на больших выборках:
агломеративная кластеризация хранит матрицу попарных расстояний размера $n \times n$.

In [ ]:
show_table(comparison_table("время, с") * 1000, digits=1, axis=None, lower_is_better=True)

Выводы по таблицам:

- метод k-средних и метод Уорда верно находят выпуклые кластеры близкого
  размера и неверно — невыпуклые кластеры и кластеры разной плотности,
  так как оба минимизируют инерцию;
- метод одиночной связи и DBSCAN находят кластеры произвольной формы, если
  кластеры разделены областями низкой плотности; метод одиночной связи
  неустойчив к отдельным удаленным точкам, результат DBSCAN зависит
  от $\varepsilon$, и одно значение $\varepsilon$ не подходит для кластеров разной плотности;
- смесь гауссовых распределений дает лучший результат на вытянутых
  кластерах и кластерах разной плотности, так как каждая компонента имеет
  собственную ковариационную матрицу; невыпуклые кластеры она, как и метод
  k-средних, не находит;
- наибольший силуэт может получить разбиение с низким ARI, например
  на кольцах: силуэт выше для компактных выпуклых кластеров независимо
  от истинного разбиения, поэтому метрики без истинного разбиения
  оценивают форму кластеров, а не соответствие задаче.

### Кластеризация изображений цифр

Для MNIST истинные классы известны, поэтому проверяют, находит ли метод
k-средних с $k = 10$ сами цифры. Сравнивают исходные пиксели и представление
из 50 главных компонент на 5000 изображениях. В таблице строки —
пространства признаков, столбцы — ARI, NMI и время.

In [ ]:
X_cluster_digits, y_cluster_digits = Xd_train[:5000], yd_train[:5000]
digit_spaces = {
    "исходные пиксели, 784 признака": X_cluster_digits,
    "PCA, 50 компонент": PCA(n_components=50, random_state=SEED).fit_transform(X_cluster_digits),
}
digit_rows, digit_labels = {}, {}
for name, space in digit_spaces.items():
    start = time.perf_counter()
    labels = KMeans(n_clusters=10, n_init=10, random_state=SEED).fit_predict(space)
    digit_rows[name] = {"ARI": adjusted_rand_score(y_cluster_digits, labels),
                        "NMI": normalized_mutual_info_score(y_cluster_digits, labels),
                        "время, с": time.perf_counter() - start}
    digit_labels[name] = labels
show_table(pd.DataFrame(digit_rows).T)

Качество в обоих пространствах практически одинаково, а в пространстве
главных компонент кластеризация выполняется быстрее. Соответствие кластеров
цифрам показывает таблица сопряженности: строки — истинные цифры,
столбцы — кластеры, в клетке — доля изображений цифры, отнесенных к кластеру.

In [ ]:
contingency = pd.crosstab(y_cluster_digits, digit_labels["PCA, 50 компонент"], normalize="index")
contingency = contingency[contingency.idxmax(axis=0).sort_values().index]
plt.figure(figsize=(8, 6))
plt.imshow(contingency.to_numpy(), cmap="Blues", vmin=0, vmax=1)
plt.xticks(range(10), contingency.columns)
plt.yticks(range(10), contingency.index)
plt.xlabel("кластер метода k-средних (k-means)")
plt.ylabel("истинная цифра")
plt.colorbar(label="доля изображений цифры")
plt.grid(False)
plt.show()

Изображения цифр 6 и 2 в основном попадают в отдельные кластеры. Изображения
цифр 0 и 1 разделены на два кластера каждая: внутри класса есть группы
изображений с разным начертанием. Цифры 4, 7 и 9 распределены по двум
общим кластерам, цифры 3, 5 и 8 также перемешаны: их изображения близки
в евклидовой метрике. Метод k-средних находит группы близких изображений,
которые не обязаны совпадать с классами.

## 6. Итоги

**Метод опорных векторов** выбирает разделяющую полосу наибольшей ширины
$\frac{2}{\|w\|}$; нарушения штрафуются hinge loss с весом $C$; границу определяют
только опорные векторы.

**Ядра.** Обучение и предсказание SVM зависят от объектов только через скалярные
произведения, поэтому их заменяют ядром $K(x, z)$ — скалярным произведением
в другом пространстве. У RBF-ядра параметр $\gamma$ обратен квадрату радиуса
влияния объекта; признаки перед ним стандартизуют.

**Метод главных компонент**: главные компоненты — собственные векторы
ковариационной матрицы, упорядоченные по убыванию собственных значений;
на практике их вычисляют через сингулярное разложение $X = U S V^\top$,
новые координаты — $U_k S_k$. Доля объясненной дисперсии определяет число
сохраняемых компонент.

**t-SNE и UMAP** строят двумерные представления, сохраняющие ближайших соседей;
параметры `perplexity` и `n_neighbors` задают размер учитываемой окрестности.
Размеры групп и расстояния между ними на таких рисунках не интерпретируют.

**Кластеризация.** Методы различаются тем, какие группы они считают кластерами:

| метод | какие кластеры находит | число кластеров | ограничение |
|---|---|---|---|
| метод k-средних | выпуклые, близкого размера | задают | невыпуклые формы, разная плотность |
| агломеративная кластеризация | зависит от способа определения расстояния | задают или выбирают по дендрограмме | память $O(n^2)$ |
| DBSCAN | области высокой плотности любой формы | определяет алгоритм | одно $\varepsilon$ для разной плотности |
| смесь гауссовых распределений | эллиптические, разного размера | задают или выбирают по BIC | невыпуклые формы |

**Метрики.** При известном разбиении — ARI, NMI, однородность, полнота, V-мера;
они сравнивают, какие объекты отнесены к одному кластеру. Без истинного
разбиения — силуэт и индексы Калински — Харабаша и Дэвиса — Болдина; они
оценивают компактность и разделенность кластеров.